<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; color: #000099;">
    <div style="text-align: center;">
        <span style="font-weight: bold; font-size: 18px; letter-spacing: 1px; text-transform: uppercase;">
        Time Series Data Processing
        </span>
        <br>
        <span style="font-weight: bold; font-size: 28px; font-style: italic; letter-spacing: 5px;">
            Load Real Time (Power Sector)
            <br>
            Demand Hot Water (Sector X - VPP Resistive Units)
        </span>
        <br>
        <span style="font-weight: bold; font-size: 18px;">
            Main Formatting Notebook
        </span>
    </div>
    <table style="margin: 20px auto; color: #000099; border-collapse: collapse; text-align: center;">
        <tr style="background-color: #E6E6E6;">
            <td style="font-size: 14px; padding: 10px 40px; opacity: 0.8;"><b>FROM</b></td>
            <td style="font-size: 14px; padding: 10px 40px; opacity: 0.8;"><b>TO</b></td>
        </tr>
        <tr style="background-color: #CCCCCC; font-size: 22px; font-style: italic; font-weight: bold;">
            <td style="padding: 5px 40px 15px 40px;">PYPSA | négaWatt</td>
            <td style="padding: 5px 40px 15px 40px;">DISPA-SET | Unleash</td>
        </tr>
    </table>
    <div style="border-top: 1px solid #000099; padding: 10px">
        This notebook processes techno-economic information extracted from <b>PyPSA</b> simulation outputs and converts the resulting infrastructure, operational, and demand-related datasets into formats compatible with the <b>Dispa-SET Unleash</b> framework.
    </div>
</div>

In [1]:
                        import os
                        import csv
from datetime           import datetime
                        import requests
                        import pandas                                 as pd
from shutil             import move
                        import numpy                                  as np
                        import shutil
from bs4                import BeautifulSoup
                        import re
                        import io
                        import plotly.graph_objects                   as go
from typing             import List, Dict, Tuple, Optional, Tuple
from IPython.display    import HTML
from difflib            import get_close_matches
from collections        import defaultdict
                        import warnings
from pathlib            import Path
                        import glob
from itertools          import permutations
from itertools          import combinations
                        import json
                        import calendar

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: right; color: #000099;">
        <span style="text-transform: uppercase;">
        <b>Auxiliary Code</b>
        </span>
            <div style="border-top: 1px solid #000099; padding: 10px">
            This cell facilitates the creation of project directories corresponding to
            </div>
        <b>ENTSO-E member countries</b>. 
        <br>
        <span style="font-style: italic; opacity: 0.8;">
        Note: Uncomment the code below only if a fresh directory structure is required.</span>
    </div>
</div>

In [2]:
"""
# List of countries with their acronyms in parentheses
countries = [
    "Albania       (AL)", "Armenia        (AM)", "Austria          (AT)", "Azerbaijan (AZ)",
    "Belarus       (BY)", "Belgium        (BE)", "Bosnia and Herz. (BA)", "Bulgaria   (BG)",
    "Croatia       (HR)", "Cyprus         (CY)", "Czech Republic   (CZ)", "Denmark    (DK)",
    "Estonia       (EE)", "Finland        (FI)", "France           (FR)", "Georgia    (GE)",
    "Germany       (DE)", "Greece         (EL)", "Hungary          (HU)", "Iceland    (IS)",
    "Ireland       (IE)", "Italy          (IT)", "Kosovo           (XK)", "Latvia     (LV)",
    "Lithuania     (LT)", "Luxembourg     (LU)", "Malta            (MT)", "Moldova    (MD)",
    "Montenegro    (ME)", "Netherlands    (NL)", "North Macedonia  (MK)", "Norway     (NO)",
    "Poland        (PL)", "Portugal       (PT)", "Romania          (RO)", "Russia     (RU)",
    "Russia Legacy (RU)", "Serbia         (RS)", "Slovakia         (SK)", "Slovenia   (SI)",
    "Spain         (ES)", "Sweden         (SE)", "Switzerland      (CH)", "Turkey     (TR)",
    "Ukraine       (UA)", "United Kingdom (UK)"
]
# Set the path where you want to create the folders
base_path = '/home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/PowerPlants'
# Ensure the base path exists
os.makedirs(base_path, exist_ok=True)
# Loop through the list of countries
for country_string in countries:
    # Use a regular expression to find the acronym inside the parentheses
    match = re.search(r'\((.*?)\)', country_string)
    # If a match is found, extract the acronym
    if match:
        acronym = match.group(1).strip()  # Use strip() to remove any extra whitespace
        folder_path = os.path.join(base_path, acronym)
        # Check if the folder already exists to avoid errors
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)
            print(f"Created folder: {folder_path}")
        else:
            print(f"Folder already exists: {folder_path}")
    else:
        print(f"Could not extract acronym from: {country_string}")
print("\nAll folders created successfully!")
"""

<>:1: SyntaxWarning: invalid escape sequence '\('
<>:1: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_2800742/3390782925.py:1: SyntaxWarning: invalid escape sequence '\('
  """


'\n# List of countries with their acronyms in parentheses\ncountries = [\n    "Albania       (AL)", "Armenia        (AM)", "Austria          (AT)", "Azerbaijan (AZ)",\n    "Belarus       (BY)", "Belgium        (BE)", "Bosnia and Herz. (BA)", "Bulgaria   (BG)",\n    "Croatia       (HR)", "Cyprus         (CY)", "Czech Republic   (CZ)", "Denmark    (DK)",\n    "Estonia       (EE)", "Finland        (FI)", "France           (FR)", "Georgia    (GE)",\n    "Germany       (DE)", "Greece         (EL)", "Hungary          (HU)", "Iceland    (IS)",\n    "Ireland       (IE)", "Italy          (IT)", "Kosovo           (XK)", "Latvia     (LV)",\n    "Lithuania     (LT)", "Luxembourg     (LU)", "Malta            (MT)", "Moldova    (MD)",\n    "Montenegro    (ME)", "Netherlands    (NL)", "North Macedonia  (MK)", "Norway     (NO)",\n    "Poland        (PL)", "Portugal       (PT)", "Romania          (RO)", "Russia     (RU)",\n    "Russia Legacy (RU)", "Serbia         (RS)", "Slovakia         (SK)", "Slove

<div style= "background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099; " >
 <span style= "font-weight: bold; font-size: 16px; " >
Section A Overview: Configuration & Environment Setup
 </span >
 <div style= "border-top: 1px solid #000099; padding: 10px; " >
This section establishes the foundational configuration for the demand processing pipeline. <br>It defines the directory structures, selects the target PyPSA scenario, configures the Virtual Power Plant (VPP) mode flag, sets the target time step (e.g., 1h), and specifies the geographical zones and target year for the simulation.
 </div >
 </div >
 <div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
    A-01. Dispa-SET Unleash Folder Path
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
    This step dynamically determines the <b>zone_folder_path</b> by locating the <i>"Dispa-SET_Unleash"</i> directory relative to your current workspace. 
    </div>
</div> 

In [3]:
### Get the directory two levels up from the current working directory
dispaSET_unleash_folder_path = Path.cwd().parent.parent
print(f"{"dispaSET_unleash_folder_path:":<55} {dispaSET_unleash_folder_path}")

dispaSET_unleash_folder_path:                           /home/ray/Dispa-SET_Unleash


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       A-02. PyPSA Source Scenario
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        There are two primary scenarios available as sources for the PyPSA raw data:<br>
    <div style="margin-left: 2em; font-size: 12px">
    <li> <b> Reference_Scenario   </b></li>
    <li> <b> Sufficiency_Scenario </b></li>
    </div>
    </div>
</div>

In [4]:
### Set the configuration
# =============================================================================
pypsa_scenario = "Reference_Scenario"
#pypsa_scenario = "Sufficiency_Scenario"
# =============================================================================
print(f"{"PyPSA Chosen Scenario:":<55} {pypsa_scenario}")

PyPSA Chosen Scenario:                                  Reference_Scenario


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        A-03. VPP (Virtual Power Plant) Mode Flag
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        The <b>VPP flag</b> controls whether the model operates in <b>Virtual Power Plant mode</b>. When enabled, additional carriers (e.g., resistive heaters) are activated to model sector-coupling and demand-side flexibility.<br>
    <b>Flag Logic:</b><br>
        <div style="margin-left: 2em; font-size: 12px">
        <li> <b> VPP = True</b>: Activates VPP mode, appending resistive heater carriers (e.g., <b>residential rural resistive heater</b>) to the <b>power_units_carriers</b> list.<br>
        <li> <b> VPP = False</b>: Disables VPP mode, limiting the model to core generation and storage technologies.<br>
        </div>
        The flag is used throughout the workflow to conditionally include sector-coupling components and enable multi-energy system modeling.<br>
        Toggling this flag allows for rapid scenario switching between a pure power-sector model and a fully integrated multi-energy system with heat, transport, and industry components.
    </div>
</div>

In [5]:
# VPP FLAG
# =============================================================================
#VPP = True
VPP = False
# =============================================================================

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
           A-04. Secondary Folders Path
        </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        The internal architecture of the <i>Dispa-SET Unleash</i> directory contains several subfolders that must be mapped to specific path variables for accurate data retrieval.
    </div>
</div>

In [6]:
### Convert the string path into a Path object
dispaSET_unleash_folder_path = Path(dispaSET_unleash_folder_path)
### Base Power Plants Data
pypsa_raw_data_folder_path = dispaSET_unleash_folder_path / "RawData_PyPSA"
print(f"{'pypsa_raw_data_folder_path:':<45} {pypsa_raw_data_folder_path}")
print("—" * 140)
### PyPSA Raw Data
loads_pypsa_raw_data_folder_path = (
    dispaSET_unleash_folder_path / "RawData_PyPSA" / pypsa_scenario / "Load_RealTime"
)
print(f"{'loads_pypsa_raw_data_folder_path:':<45} {loads_pypsa_raw_data_folder_path}")
print("—" * 140)
### PyPSA Formatted Data
loads_pypsa_formated_data_folder_path = (
    dispaSET_unleash_folder_path / "Database_PyPSA" / pypsa_scenario / "Load_RealTime"
)
print(f"{'loads_pypsa_formated_data_folder_path:':<45} {loads_pypsa_formated_data_folder_path}")
print("—" * 140)
### PyPSA Formatted Data with VPP
vpp_scenario_folder = f"{pypsa_scenario}________VPP" 
vpp_loads_pypsa_formated_data_folder_path = (
    dispaSET_unleash_folder_path / "Database_PyPSA" / vpp_scenario_folder / "Load_RealTime"
)
print(f"{'vpp_loads_pypsa_formated_data_folder_path:':<45} {vpp_loads_pypsa_formated_data_folder_path}")
print("—" * 140)
### PyPSA VPP Features Data
vpp_features_data_folder_path = (
    dispaSET_unleash_folder_path / "Database_PyPSA" / vpp_scenario_folder / "VPP_data" / "VPP_features"
)
print(f"{'vpp_features_data_folder_path:':<45} {vpp_features_data_folder_path}")
print("—" * 140)
### PyPSA VPP Demand Data
vpp_demand_data_folder_path = (
    dispaSET_unleash_folder_path / "Database_PyPSA" / vpp_scenario_folder / "VPP_data" / "VPP_demand"
)
print(f"{'vpp_demand_data_folder_path:':<45} {vpp_demand_data_folder_path}")
# PyPSA Data Formatted Data with VPP
#vpp_scenario_folder = f"{pypsa_scenario}________VPP" 
#vpp_data_pypsa_formated_data_folder_path = (
#    dispaSET_unleash_folder_path / "Database_PyPSA" / vpp_scenario_folder / "VPP_data"
#)
#print(f"{'vpp_data_pypsa_formated_data_folder_path:':<55} {vpp_data_pypsa_formated_data_folder_path}")
#print("—" * 140)

pypsa_raw_data_folder_path:                   /home/ray/Dispa-SET_Unleash/RawData_PyPSA
————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
loads_pypsa_raw_data_folder_path:             /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/Load_RealTime
————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
loads_pypsa_formated_data_folder_path:        /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/Load_RealTime
————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
vpp_loads_pypsa_formated_data_folder_path:    /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario________VPP/Load_RealTime
—————————————————————————————————————————————————————————————————————————————————————————————————————————————————

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       A-05. Dispa-SET Time Step
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        The time series data must be resampled to a predetermined time step.
        <br>
        The UNLEASH project utilizes three levels of granularity:
        <div style="margin-left: 2em; font-size: 12px">
            <li>One hour (1h)</li>
            <li>Thirty minutes (30min)</li>
            <li>Fifteen minutes (15min)</li>
        </div>
    </div>
</div>

In [7]:
### Set the time step to which data is formating to 
data_target_time_step = '1h'
# data_target_time_step = '15min'
# data_target_time_step = '30min'
print(f"{"Selected time step:":<55} {data_target_time_step}")

Selected time step:                                     1h


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        A-06. Zone(s) Configuration
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        Define the target geographical zone(s) for data processing. This selection determines which regional datasets will be accessed and where the resulting formatted files will be stored.
        <br>
        Use <b>ISO 3166-1 alpha-2</b> standard codes for European countries (e.g., <i>AT, BE, BG, CH, DE, FR</i>). 
    </div>
</div>

In [8]:
# =============================================================================
all_zones = [
    "AL", "AM", "AT", "AZ", "BY", "BE", "BA", "BG", "HR", "CY", "CZ", "DK",
    "EE", "FI", "FR", "GE", "DE", "EL", "HU", "IS", "IE", "IT", "XK", "LV",
    "LT", "LU", "MT", "MD", "ME", "NL", "MK", "NO", "PL", "PT", "RO", "RU",
    "RS", "SK", "SI", "ES", "SE", "CH", "TR", "UA", "UK"
]
# =============================================================================
active_selection = {"BE", "FR", "DE", "NL", "UK"}
# =============================================================================
vpp_active_selection = {"BE"}
# =============================================================================
filter_zones = lambda selected: [z for z in all_zones if z in selected]
zone_names = filter_zones(active_selection)
vpp_zone_names = filter_zones(vpp_active_selection)
print("Selected Zones:", zone_names)
print("VPP Selected Zones:", vpp_zone_names)

Selected Zones: ['BE', 'FR', 'DE', 'NL', 'UK']
VPP Selected Zones: ['BE']


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: right; color: #000099;">
    <span style="text-transform: uppercase;">
    <b>International Nomenclature</b>
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px">
        The following dictionary maps <b>standardized country codes</b> to their various synonyms, aliases, or alternative naming conventions used across international databases
    <br>
    (e.g., ENTSO-E, Eurostat, or ISO variants).
    <br>
    <b>Purpose:</b> This ensures the script can correctly identify and link data even when source files use inconsistent regional identifiers (e.g., <i>UK</i> vs. <i>GB</i> or <i>EL</i> vs. <i>GR</i>).
    </div>
</div>

In [9]:
### Define a clean base dictionary (no messy padding or unnecessary lists)
# =============================================================================
raw_countries = {
    "AL": ("Albania",            ""),  "AM": ("Armenia",          ""),  "AT": ("Austria",          ""),
    "AZ": ("Azerbaijan",         ""),  "BY": ("Belarus",          ""),  "BE": ("Belgium",          ""),
    "BA": ("Bosnia and Herz.",   ""),  "BG": ("Bulgaria",         ""),  "HR": ("Croatia",          ""),
    "CY": ("Cyprus",             ""),  "CZ": ("Czech Republic",   ""),  "DK": ("Denmark",          ""),
    "EE": ("Estonia",            ""),  "FI": ("Finland",          ""),  "FR": ("France",           ""),
    "GE": ("Georgia",            ""),  "DE": ("Germany",          ""),  "EL": ("Greece",         "GR"),
    "HU": ("Hungary",            ""),  "IS": ("Iceland",          ""),  "IE": ("Ireland",          ""),
    "IT": ("Italy",              ""),  "XK": ("Kosovo",           ""),  "LV": ("Latvia",           ""),
    "LT": ("Lithuania",          ""),  "LU": ("Luxembourg",       ""),  "MT": ("Malta",            ""),
    "MD": ("Moldova",            ""),  "ME": ("Montenegro",       ""),  "NL": ("Netherlands",      ""),
    "MK": ("North Macedonia",    ""),  "NO": ("Norway",           ""),  "PL": ("Poland",           ""),
    "PT": ("Portugal",           ""),  "RO": ("Romania",          ""),  "RU": ("Russia",           ""),
    "RS": ("Serbia",             ""),  "SK": ("Slovakia",         ""),  "SI": ("Slovenia",         ""),
    "ES": ("Spain",              ""),  "SE": ("Sweden",           ""),  "CH": ("Switzerland",      ""),
    "TR": ("Turkey",             ""),  "UA": ("Ukraine",          ""),  "UK": ("United Kingdom", "GB")
}
# =============================================================================
### Constructing dicctionary
zone_names_equivalences_dict = {
    code: {"Acronym": [acronym if acronym else " "], "name": [name]}
    for code, (name, acronym) in raw_countries.items()
}
### Filter the dictionary using only the keys present in zone_names
selected_zone_names_equivalences_dict = {
    k: zone_names_equivalences_dict[k] 
    for k in zone_names 
    if k in zone_names_equivalences_dict
}
### Print only the zones that have a valid alternative acronym
for key, value in selected_zone_names_equivalences_dict.items():
    acronym = value["Acronym"][0].strip()
    if acronym:  # Evaluates to True only if the acronym is not empty or spaces
        print(f"Key: {key};    Acronym: {value['Acronym']};    Name: {value['name']}\n")

Key: UK;    Acronym: ['GB'];    Name: ['United Kingdom']



<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        A-07. Data Reference Year
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
            Define the <b>target temporal scope</b> for the processing workflow. This variable determines which yearly dataset will be filtered, formatted, and exported.
    </div>
</div>

In [10]:
### Year to which data is formatting to
# =============================================================================
data_target_year = "2030"
#data_target_year = "2040"
#data_target_year = "2050"
# =============================================================================
print(f"Selected Year: {data_target_year}")

Selected Year: 2030


<div style= "background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099; " >
 <span style= "font-weight: bold; font-size: 16px; " >
Section B Overview: Electric Load Extraction & Aggregation
 </span >
 <div style= "border-top: 1px solid #000099; padding: 10px; " >
This section handles the initial extraction of baseline electricity demands.<br> It loads the raw PyPSA load time series, splits them by country using international nomenclature equivalences, extracts specific node-level electric loads (including agriculture and industry), and aggregates them into a single total electric load profile per country.
 </div >
 </div >
<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       B-01. Loading and Splitting Load Time Series by Country
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process loads the <b>load time series</b> CSV file and splits the columns into separate DataFrames for each country.<br> It uses country codes and their <b>acronym equivalents</b> (e.g., <b>UK ↔ GB</b>) to match the correct columns.<br>
    <b>Loading Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        The script constructs the file path: <b>{loads_pypsa_raw_data_folder_path}/loads_{scenario}_{year}.csv</b>.
            <br>If the file exists, it loads the full time series with the first column as index. 
            <br>It then iterates through each zone, using both the primary code and its acronym to match columns in the DataFrame. Matching columns are stored in a dictionary keyed by the zone code.
        </div>
    </div>
</div>

In [11]:
### Filter dictionary for requested zones
selected_zone_names_equivalences_dict = {
    k: zone_names_equivalences_dict[k] 
    for k in zone_names 
    if k in zone_names_equivalences_dict
}
### 3. Construct target CSV file path using Path / operator
file_path = loads_pypsa_raw_data_folder_path / f"loads_{pypsa_scenario}_{data_target_year}.csv"
### Initialize single-level dictionary: raw_data_dict["UK"] = DataFrame
raw_data_dict = {}
if file_path.exists():
    ### Load full load time series
    df = pd.read_csv(file_path, index_col=0)
    ### 4. Split columns into country dictionary
    for zone_key, metadata in selected_zone_names_equivalences_dict.items():
        ### Search patterns for primary code (e.g. "UK") and acronym (e.g. "GB")
        search_codes = [zone_key]
        alt_acronym = metadata["Acronym"][0].strip()
        if alt_acronym:
            search_codes.append(alt_acronym)
        ### Match columns containing any of the country codes
        matching_cols = [
            col for col in df.columns 
            if any(col.startswith(code) or f"_{code}" in col or f" {code}" in col for code in search_codes)
        ]
        if matching_cols:
            raw_data_dict[zone_key] = df[matching_cols].copy()
        else:
            print(f"⚠️ No matching columns found for '{zone_key}' ({search_codes})")
    print(f"✅ `raw_data_dict` populated successfully from:\n   {file_path}\n")
    ### Check and print dictionary population status
    if raw_data_dict:
        print(f"📊 Dictionary Summary ({len(raw_data_dict)} zones populated):")
        print("-" * 50)
        for zone, data in raw_data_dict.items():
            print(f"  • Zone: {zone:<4} | Columns: {len(data.columns):<3} | Shape: {data.shape}")
        print("-" * 50)
    else:
        print("⚠️ Warning: `raw_data_dict` is empty! No matching columns were found for any zone.")
else:
    print(f"❌ File not found at: {file_path}")

✅ `raw_data_dict` populated successfully from:
   /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/Load_RealTime/loads_Reference_Scenario_2030.csv

📊 Dictionary Summary (5 zones populated):
--------------------------------------------------
  • Zone: BE   | Columns: 23  | Shape: (8760, 23)
  • Zone: FR   | Columns: 23  | Shape: (8760, 23)
  • Zone: DE   | Columns: 23  | Shape: (8760, 23)
  • Zone: NL   | Columns: 23  | Shape: (8760, 23)
  • Zone: UK   | Columns: 46  | Shape: (8760, 46)
--------------------------------------------------


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
     B-02. Extracting Electric Load Data by Node and Technology
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process extracts <b>electric load time series</b> from the raw load data, organized by <b>node</b> and <b>technology type</b>.<br> It separates base electricity loads from sector-specific loads (e.g., agriculture, industry) and creates a structured dictionary of DataFrames per country.<br>
    <b>Extraction Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each country, the script identifies unique node prefixes (e.g., <b>"BE1 0"</b>, <b>"BE1 1"</b>). For each node, it extracts:
        <div style="margin-left: 2em;">
            <li><b>Base Electricity Load</b> → Exact node match (e.g., <b>"BE1 0"</b>)<br>
            <li><b>Sector-Specific Loads</b> → Columns starting with the node and ending with specific technology suffixes (e.g., <b>"agriculture electricity"</b>, <b>"industry electricity"</b>)
        </div>
        Extracted columns are renamed with a <b>_low voltage</b> suffix for consistency.
    </div>
</div>

In [12]:
### 1. Define the suffix-based technologies to extract
# =============================================================================
load_technologies = [
    "agriculture electricity",
    "industry electricity"
]
# =============================================================================
### Initialize single-level dictionary: electric_loads["BE"] = DataFrame
electric_loads = {}
### 2. Iterate through each country DataFrame in raw_data_dict
for zone_key, df_zone in raw_data_dict.items():
    if df_zone.empty:
        continue
    extracted_columns = {}
    ### Extract unique node prefixes for THIS specific country (e.g., 'BE1 0', 'BE1 1')
    nodes = df_zone.columns.str.split(" ").str[:2].str.join(" ").unique()
    for node in nodes:
        ### --- 1. Extract Base Electricity Load (Exact node match, e.g., "BE1 0") ---
        if node in df_zone.columns:
            col_name = f"{node}_low voltage"
            extracted_columns[col_name] = df_zone[node]
        ### --- 2. Extract Sector-Specific Technologies (Prefix + Suffix match) ---
        for tech in load_technologies:
            matching_columns = [
                col for col in df_zone.columns 
                if col.startswith(node) and col.endswith(tech)
            ]
            if matching_columns:
                original_col = matching_columns[0]
                col_name = f"{node} {tech}_low voltage"
                extracted_columns[col_name] = df_zone[original_col]   
    ### If matching profiles were found, build the country DataFrame
    if extracted_columns:
        electric_loads[zone_key] = pd.DataFrame(extracted_columns, index=df_zone.index)
### Summary Check
print("✅ `electric_loads` dictionary created successfully.\n")
print("📊 Summary of extracted DataFrames per country:")
print("-" * 65)
for zone, df_res in electric_loads.items():
    print(f"  • Country: {zone:<4} | Columns: {len(df_res.columns):<3} | Shape: {df_res.shape}")
    print(f"    Sample Columns: {list(df_res.columns[:3])} ...")
print("-" * 65)

✅ `electric_loads` dictionary created successfully.

📊 Summary of extracted DataFrames per country:
-----------------------------------------------------------------
  • Country: BE   | Columns: 3   | Shape: (8760, 3)
    Sample Columns: ['BE1 0_low voltage', 'BE1 0 agriculture electricity_low voltage', 'BE1 0 industry electricity_low voltage'] ...
  • Country: FR   | Columns: 3   | Shape: (8760, 3)
    Sample Columns: ['FR1 0_low voltage', 'FR1 0 agriculture electricity_low voltage', 'FR1 0 industry electricity_low voltage'] ...
  • Country: DE   | Columns: 3   | Shape: (8760, 3)
    Sample Columns: ['DE1 0_low voltage', 'DE1 0 agriculture electricity_low voltage', 'DE1 0 industry electricity_low voltage'] ...
  • Country: NL   | Columns: 3   | Shape: (8760, 3)
    Sample Columns: ['NL1 0_low voltage', 'NL1 0 agriculture electricity_low voltage', 'NL1 0 industry electricity_low voltage'] ...
  • Country: UK   | Columns: 6   | Shape: (8760, 6)
    Sample Columns: ['GB0 0_low voltage', 

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        B-03. Summing All Electric Load Columns
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process aggregates all electric load columns for each country into a <b>single total time series</b>.<br>
    The resulting DataFrame contains one column per country (<b>total_electric_load</b>) representing the <b>total electric load</b> at each snapshot.<br>
    <b>Aggregation Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each country, the script sums all columns of the DataFrame <b>horizontally</b> (along the column axis) using <b>df.sum(axis=1)</b>. <br>
        This produces a single Series where each value is the sum of all electric load components at that snapshot. <br>
        The Series is then converted to a DataFrame with a single column named <b>"total_electric_load"</b>, preserving the snapshot as the index.<br>
        The resulting dictionary, <b>total_electric_loads</b>, contains one DataFrame per country with a single column representing the total electric load. This aggregated profile is ready for Dispa-SET load formatting.
    </div>
</div>

In [13]:
total_electric_loads = {}
for country, df in electric_loads.items():
    total_load = df.sum(axis=1)
    total_electric_loads[country] = total_load.to_frame(
        name="total_electric_load"
    )
    total_electric_loads[country].index.name = "snapshot"
print("\nProcess completed.")
for country, df in total_electric_loads.items():
    print(
        f"{country}: {len(df)} snapshots processed, "
        f"column = '{df.columns[0]}'"
        )


Process completed.
BE: 8760 snapshots processed, column = 'total_electric_load'
FR: 8760 snapshots processed, column = 'total_electric_load'
DE: 8760 snapshots processed, column = 'total_electric_load'
NL: 8760 snapshots processed, column = 'total_electric_load'
UK: 8760 snapshots processed, column = 'total_electric_load'


<div style= "background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099; " >
 <span style= "font-weight: bold; font-size: 16px; " >
Section C Overview: Generators & Links Electric Consumption
 </span >
 <div style= "border-top: 1px solid #000099; padding: 10px; " >
This section processes the electric consumption profiles from generators and multi-bus links (e.g., EV chargers, batteries, H2 electrolysis, DAC). <br>It correctly assigns normal and reversed inter-node pipeline flows to their respective zones, filters by active technologies, and calculates the net total electric consumption by summing positive loads and subtracting generation/injections.
 </div >
 </div >
<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
      C-01.  Loading Generators & Links Electric Consumption (with Automatic Date Parsing)
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process loads the <b>generators and links electric consumption</b> CSV file with automatic date parsing.<br> The <b>"snapshot"</b> column is explicitly parsed as dates and set as the DataFrame index, ensuring that the time series data is properly formatted for time-based operations.<br>
    <b>Loading Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        The script constructs the filename: <b>generators_links_electric_consumption_{scenario}_{year}.csv</b>. 
        <br>
        It uses <b>Path</b> for cross-platform path handling, loads the CSV with <b>parse_dates=["snapshot"]</b> to automatically convert the snapshot column to datetime, and sets it as the index with <b>index_col="snapshot"</b>.
    </div>
</div>

In [14]:
### LOAD ALL GENERATORS/LINKS ELECTRIC CONSUMPTION
# _____________________________________________________
### Construct the filename dynamicallyfile_path = (
file_path = (    
    Path(loads_pypsa_raw_data_folder_path)
    / f"generators_links_electric_consumption_{pypsa_scenario}_{data_target_year}.csv"
)
generators_links_electric_consumption = pd.read_csv(
    file_path,
    parse_dates=["snapshot"],
    index_col="snapshot"
    )
print(
    f"✅ Successfully loaded DataFrame from: {file_path}"
)
print(
    f"Shape: {generators_links_electric_consumption.shape}"
)
print(
    generators_links_electric_consumption.head()
)

✅ Successfully loaded DataFrame from: /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/Load_RealTime/generators_links_electric_consumption_Reference_Scenario_2030.csv
Shape: (8760, 261)
                     BE1 0 V2G  BE1 0 EV charger  BE1 0 battery charger-2030  \
snapshot                                                                       
2013-01-01 00:00:00   0.017559        244.177616                 3903.507131   
2013-01-01 01:00:00   0.017594        184.364037                 4508.084838   
2013-01-01 02:00:00   0.017605        172.481731                 4054.868880   
2013-01-01 03:00:00   0.017497        217.937680                 2223.847837   
2013-01-01 04:00:00   0.017989        417.068919                  986.285253   

                     BE1 0 battery discharger-2030  \
snapshot                                             
2013-01-01 00:00:00                       0.288390   
2013-01-01 01:00:00                       0.293494   
2013-01-01 02:00:00      

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
      C-02. Building Country-Level Dictionary with Normal and Reversed Inter-Node Links
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process creates a <b>country-level dictionary</b> of generator and link electric consumption data, with advanced handling for <b>normal</b> and <b>reversed</b> inter-node links (e.g., gas pipelines, H2 pipelines). <br>It identifies all nodes in a column header, determines ownership based on the column's direction, and assigns the column to the correct zone.<br>
    <b>Assignment Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each column, the script identifies <b>all nodes</b> appearing in the header and preserves their order. It then determines ownership based on the column type:
            <div style="margin-left: 2em;">
            <li><b>Normal Column</b> (no "reversed" in name) → <b>First node</b> owns the column (source node).<br>
            <li><b>Reversed Column</b> (contains "reversed") → <b>Second node</b> owns the column (destination node in reverse flow).
            </div>   
        This ensures that bi-directional pipeline flows are correctly assigned to the appropriate zone.
    </div>
</div>

In [15]:
### MASTER DICTIONARY
generators_links_electric_consumption_dict = {}
# GET ALL COLUMN NAMES
all_columns = generators_links_electric_consumption.columns
# BUILD ZONE PREFIXES
zone_prefixes_dict = {}
for zone in zone_names:
    prefixes = [zone]
    if zone in zone_names_equivalences_dict:
        alt_acronym = (
            zone_names_equivalences_dict[zone]["Acronym"][0]
            .strip()
        )
        if alt_acronym:
            prefixes.append(alt_acronym)
    zone_prefixes_dict[zone] = prefixes
# SPLIT COLUMNS BY ZONE
for zone in zone_names:
    # Get valid prefixes for this zone
    prefixes = zone_prefixes_dict[zone]
    matching_cols = []
    # CHECK EVERY COLUMN
    for col in all_columns:
        col_str = str(col)
        # CASE 1:
        # COLUMN STARTS WITH THIS ZONE/PREFIX
        starts_with_zone = any(
            col_str.startswith(prefix)
            for prefix in prefixes
        )
        if starts_with_zone:
            matching_cols.append(col)
            continue
        # CASE 2:
        # COLUMN DOES NOT START WITH A ZONE
        # Find ALL nodes appearing in the column header
        # and preserve their order.
        found_nodes = []
        for candidate_zone in zone_names:
            candidate_prefixes = zone_prefixes_dict[
                candidate_zone
            ]
            for prefix in candidate_prefixes:
                # Search for every occurrence of this
                # prefix in the column name
                search_start = 0
                while True:
                    position = col_str.find(
                        prefix,
                        search_start
                    )
                    if position == -1:
                        break
                    # Make sure the prefix is followed by
                    # the expected node structure.
                    if (
                        position + len(prefix)
                        < len(col_str)
                    ):
                        next_character = col_str[
                            position + len(prefix)
                        ]
                        if not (
                            next_character == " "
                            or next_character.isdigit()
                        ):
                            search_start = (
                                position + len(prefix)
                            )
                            continue
                    # Store node and its position
                    found_nodes.append(
                        (
                            position,
                            candidate_zone
                        )
                    )
                    # Continue searching after this
                    # occurrence
                    search_start = (
                        position + len(prefix)
                    )
        # SORT NODES BY THEIR POSITION IN THE HEADER
        found_nodes.sort(
            key=lambda x: x[0]
        )
        # Remove duplicate detections at the same
        # position, keeping the first one
        ordered_zones = []
        previous_position = None
        for position, found_zone in found_nodes:
            if position != previous_position:
                ordered_zones.append(found_zone)
                previous_position = position
        # DETERMINE WHICH NODE OWNS THE COLUMN
        column_lower = col_str.lower()
        # NORMAL COLUMN
        # First node owns the column.
        # Example:
        # H2 pipeline BE1 0 -> NL1 0-2030
        #        BE  ->  NL
        #        ↑
        #      OWNER
        if "reversed" not in column_lower:
            if len(ordered_zones) >= 1:
                selected_zone = ordered_zones[0]
            else:
                selected_zone = None
        # REVERSED COLUMN
        # Second node owns the column.
        # Example:
        # H2 pipeline BE1 0 -> NL1 0-reversed-2030
        #        BE  ->  NL
        #              ↑
        #            OWNER
        else:
            if len(ordered_zones) >= 2:
                selected_zone = ordered_zones[1]
            else:
                selected_zone = None
        # ASSIGN COLUMN TO CURRENT ZONE
        if selected_zone == zone:
            matching_cols.append(col)
    # CREATE ZONE DATAFRAME
    if matching_cols:
        generators_links_electric_consumption_dict[
            zone
        ] = (
            generators_links_electric_consumption[
                matching_cols
            ].copy()
        )
    else:
        generators_links_electric_consumption_dict[
            zone
        ] = pd.DataFrame(
            index=generators_links_electric_consumption.index
        )
# INSPECTION
print(
    "✅ Successfully built country-level dictionary "
    "with normal and reversed inter-node links!\n"
)
for zone, zone_df in (
    generators_links_electric_consumption_dict.items()
):
    print(
        f"  • Zone key: '{zone}' | "
        f"Columns: {len(zone_df.columns):<3} | "
        f"Shape: {zone_df.shape}"
    )
# CHECK PIPELINES
print("\n" + "=" * 70)
print("PIPELINE ASSIGNMENT CHECK")
print("=" * 70)
for zone, zone_df in (
    generators_links_electric_consumption_dict.items()
):
    pipeline_columns = [
        col
        for col in zone_df.columns
        if "pipeline" in str(col).lower()
    ]
    if pipeline_columns:
        print(f"\n{zone}:")
        for col in pipeline_columns:
            print(f"  {col}")

✅ Successfully built country-level dictionary with normal and reversed inter-node links!

  • Zone key: 'BE' | Columns: 44  | Shape: (8760, 44)
  • Zone key: 'FR' | Columns: 48  | Shape: (8760, 48)
  • Zone key: 'DE' | Columns: 43  | Shape: (8760, 43)
  • Zone key: 'NL' | Columns: 47  | Shape: (8760, 47)
  • Zone key: 'UK' | Columns: 79  | Shape: (8760, 79)

PIPELINE ASSIGNMENT CHECK

BE:
  gas pipeline BE1 0 -> FR1 0-2030
  gas pipeline BE1 0 -> NL1 0-2030
  gas pipeline BE1 0 <-> FR1 0-2030
  gas pipeline BE1 0 <-> NL1 0-2030
  gas pipeline DE1 0 -> BE1 0-reversed-2030
  gas pipeline FR1 0 <-> BE1 0-reversed-2030
  gas pipeline GB0 0 <-> BE1 0-reversed-2030
  gas pipeline NL1 0 -> BE1 0-reversed-2030
  H2 pipeline BE1 0 -> DE1 0-2030
  H2 pipeline BE1 0 -> FR1 0-2030
  H2 pipeline BE1 0 -> GB0 0-2030
  H2 pipeline BE1 0 -> NL1 0-2030

FR:
  gas pipeline FR1 0 -> DE1 0-2030
  gas pipeline FR1 0 <-> BE1 0-2030
  gas pipeline BE1 0 -> FR1 0-reversed-2030
  gas pipeline BE1 0 <-> FR1 0-r

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        C-03. Link Technologies (with Commented-Out Heat Pumps and Resistive Heaters)
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process defines the <b>link technologies</b> used in the model. In this version, <b>heat pumps and resistive heaters have been commented out</b>, meaning they are excluded from the link technologies list. Only the core technologies (V2G, EV chargers, battery chargers/dischargers, gas pipelines, H2 technologies, and DAC) are included.<br>
    <b>Filtering Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
The script defines a <b>base list</b> of link technologies that excludes both heat pumps and resistive heaters. The active list includes only:
        <div style="margin-left: 2em;">
        <li><b>Vehicle-to-Grid:</b> V2G<br>
        <li><b>Electric Vehicles:</b> EV charger<br>
        <li><b>Battery Storage:</b> battery charger, battery discharger<br>
        <li><b>Gas Pipelines:</b> gas pipeline, gas pipeline new<br>
        <li><b>Synthetic Fuels:</b> methanolisation, Haber-Bosch<br>
        <li><b>Hydrogen:</b> H2 Electrolysis, H2 pipeline, H2 pipeline retrofitted<br>
        <li><b>Carbon Capture:</b> DAC
        </div>
        <b>All heat pumps</b> (air and ground heat pumps) are commented out and excluded.
        <br>The conditional addition of resistive heaters has also been commented out, so they are excluded regardless of VPP mode.
    </div>
</div>

In [16]:
### TECHNOLOGY LISTS
# =====================================================
link_technologies = [
    "V2G",
    "EV charger",
    "battery charger",
    "battery discharger",
    "gas pipeline",
    "gas pipeline new",
    "methanolisation",
    "Haber-Bosch",
    "H2 Electrolysis",
    "H2 pipeline",
    "H2 pipeline retrofitted",
    "DAC",
    #"urban central air heat pump",
    #"services urban decentral air heat pump",
    #"services rural air heat pump",
    #"services rural ground heat pump",
    #"residential rural air heat pump",
    #"residential rural ground heat pump",
    #"residential urban decentral air heat pump",
    ]
'''if not VPP:
    link_technologies += [
            "urban central resistive heater",
            "services urban decentral resistive heater",
            "services rural resistive heater",
            "residential rural resistive heater",
            "residential urban decentral resistive heater",
        ]'''
# =====================================================
generator_technologies = ["load"]
print(f"Number of link technologies: {len(link_technologies)}")
print("Link technologies:")
for tech in link_technologies:
    print(f"  - {tech}")

Number of link technologies: 12
Link technologies:
  - V2G
  - EV charger
  - battery charger
  - battery discharger
  - gas pipeline
  - gas pipeline new
  - methanolisation
  - Haber-Bosch
  - H2 Electrolysis
  - H2 pipeline
  - H2 pipeline retrofitted
  - DAC


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        C-04. Filtering Generator and Link Electric Consumption Columns by Technology
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process filters the generator and link electric consumption DataFrames to retain only columns that match the <b>technology lists</b> (link_technologies and generator_technologies). It removes unwanted columns, converts the snapshot to a DatetimeIndex, and stores the filtered DataFrames in a new dictionary.<br>
    <b>Filtering Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each zone, the script performs the following steps:
            <div style="margin-left: 2em;">
        <b>1.</b> Removes any unwanted column named <b>"0"</b> (accidentally imported from CSV)<br>
        <b>2.</b> Identifies the snapshot (either as a column or as the index)<br>
        <b>3.</b> Filters columns to retain only those containing any technology name from <b>technologies_to_select</b><br>
        <b>4.</b> Converts the snapshot to a <b>DatetimeIndex</b> and sets it as the DataFrame index<br>
        <b>5.</b> Stores the cleaned DataFrame in a new dictionary
            </div>
        </div>
    </div>
</div>

In [17]:
# =====================================================
### COMBINE TECHNOLOGY LISTS
technologies_to_select = (
    link_technologies
    + generator_technologies
)
### CREATE NEW NESTED DICTIONARY
generators_links_electric_consumption_dict_selected = {}
### FILTER EACH ZONE DATAFRAME
for zone, df in (
    generators_links_electric_consumption_dict.items()
):
    print(f"\n--- Processing {zone} ---")
    ### REMOVE UNWANTED INDEX COLUMN IF IT EXISTS
    ### Remove a column named "0" if it was accidentally
    ### imported from a previous CSV export.
    df_clean = df.drop(
        columns=["0"],
        errors="ignore"
    ).copy()
    ### IDENTIFY SNAPSHOT
    if "snapshot" in df_clean.columns:
        ### Snapshot is a regular column
        snapshot_data = df_clean["snapshot"]
        data_columns = [
            column
            for column in df_clean.columns
            if column != "snapshot"
        ]
    else:
        ### Snapshot is already the DataFrame index
        snapshot_data = df_clean.index
        data_columns = df_clean.columns.tolist()
    ### SELECT TECHNOLOGY COLUMNS
    selected_columns = []
    for column in data_columns:
        column_name = str(column)
        if any(
            technology in column_name
            for technology in technologies_to_select
        ):
            selected_columns.append(column)
    ### BUILD SELECTED DATAFRAME
    selected_df = df_clean[
        selected_columns
    ].copy()
    ### KEEP SNAPSHOT AS INDEX
    if "snapshot" in df_clean.columns:
        selected_df.index = pd.to_datetime(
            snapshot_data
        )
    else:
        selected_df.index = snapshot_data
    selected_df.index.name = "snapshot"
    ### STORE IN NEW DICTIONARY
    generators_links_electric_consumption_dict_selected[
        zone
    ] = selected_df
    ### INSPECTION
    print(
        f"  Selected asset columns: "
        f"{len(selected_columns)}"
    )
    print(
        f"  Final shape: "
        f"{selected_df.shape}"
    )
### FINAL INSPECTION
print("\n" + "=" * 70)
print(
    "SUCCESSFULLY CREATED:"
)
print(
    "'generators_links_electric_consumption_dict_selected'"
)
print("=" * 70)
print(
    f"Total zones: "
    f"{len(generators_links_electric_consumption_dict_selected)}"
)
for zone, df in (
    generators_links_electric_consumption_dict_selected.items()
):
    print("\n" + "-" * 60)
    print(f"Zone: {zone}")
    print(
        f"Shape: {df.shape}"
    )
    print(
        f"Number of asset columns: "
        f"{len(df.columns)}"
    )
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nFirst 2 rows:")
    print(df.head(2))


--- Processing BE ---
  Selected asset columns: 22
  Final shape: (8760, 22)

--- Processing FR ---
  Selected asset columns: 18
  Final shape: (8760, 18)

--- Processing DE ---
  Selected asset columns: 21
  Final shape: (8760, 21)

--- Processing NL ---
  Selected asset columns: 23
  Final shape: (8760, 23)

--- Processing UK ---
  Selected asset columns: 32
  Final shape: (8760, 32)

SUCCESSFULLY CREATED:
'generators_links_electric_consumption_dict_selected'
Total zones: 5

------------------------------------------------------------
Zone: BE
Shape: (8760, 22)
Number of asset columns: 22

Columns:
['BE1 0 V2G', 'BE1 0 EV charger', 'BE1 0 battery charger-2030', 'BE1 0 battery discharger-2030', 'gas pipeline BE1 0 -> FR1 0-2030', 'gas pipeline BE1 0 -> NL1 0-2030', 'gas pipeline BE1 0 <-> FR1 0-2030', 'gas pipeline BE1 0 <-> NL1 0-2030', 'gas pipeline DE1 0 -> BE1 0-reversed-2030', 'gas pipeline FR1 0 <-> BE1 0-reversed-2030', 'gas pipeline GB0 0 <-> BE1 0-reversed-2030', 'gas pipeli

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        C-05. Calculating Total Electric Consumption (with Commented-Out Positive Technologies)
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process calculates the <b>total electric consumption</b> for each zone by summing positive contributions (consumption) and subtracting negative contributions (generation or injection). In this version, <b>heat pumps and resistive heaters have been commented out</b> from the positive technologies list, meaning they are not included in the total electric consumption calculation.<br>
    <b>Calculation Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        The script categorizes technologies into <b>positive</b> (consuming electricity) and <b>negative</b> (generating or injecting electricity). <br>The positive technologies list currently includes only EV chargers, battery chargers, gas pipelines, H2 technologies, and DAC.<br> <b>Heat pumps and resistive heaters are commented out</b> and therefore excluded from the calculation.
        <br>
        For each zone, it iterates through all technology categories, sums the values of all matching columns, and applies the appropriate sign:
            <div style="text-align: center; font-size:14px;">        
        <b>Total = Σ(Positive Technologies) - Σ(Negative Technologies)</b>
            </div>        
        The results are stored in a <b>new dictionary</b> (<b>total_electric_consumption</b>) with a single column named <b>"Total_Electric_Consumption"</b> per zone.<br>
        A warning is issued if any columns remain unassigned.
    </div>
</div>

In [18]:
### CALCULATE TOTAL ELECTRIC CONSUMPTION
# _____________________________________________________
### Technologies with POSITIVE contribution
# =====================================================
positive_technologies = [
    "EV charger",
    "battery charger",
    "gas pipeline",
    "gas pipeline new",
    "methanolisation",
    "Haber-Bosch",
    "H2 Electrolysis",
    "H2 pipeline",
    "urban central DAC",
    #"urban central air heat pump",
    #"services urban decentral air heat pump",
    #"services rural air heat pump",
    #"services rural ground heat pump",
    #"residential rural air heat pump",
    #"residential rural ground heat pump",
    #"residential urban decentral air heat pump",
]
'''if not VPP:
    positive_technologies += [
        "urban central resistive heater",
        "services urban decentral resistive heater",
        "services rural resistive heater",
        "residential rural resistive heater",
        "residential urban decentral resistive heater",
]'''
# =====================================================
### Technologies with NEGATIVE contribution
# =====================================================
negative_technologies = [
    "V2G",
    "battery discharger",
    # "H2 turbine",
    # "H2 Fuel Cell",
    "load",
    "low voltage load",
]
# =====================================================
### CREATE NEW DICTIONARY
total_electric_consumption = {}
### APPLY FORMULA TO EVERY DATAFRAME
for zone, df in generators_links_electric_consumption_dict_selected.items():
    print(f"\n--- Processing {zone} ---")
    ### Start with zero
    total_electric_consumption_series = pd.Series(
        0.0,
        index=df.index
    )
    ### Keep track of columns already assigned
    assigned_columns = set()
    ### POSITIVE CONTRIBUTIONS
    for technology in positive_technologies:
        matching_columns = [
            column
            for column in df.columns
            if (
                technology in str(column)
                and column not in assigned_columns
            )
        ]
        if matching_columns:
            print(
                f"  + {technology}: "
                f"{len(matching_columns)} column(s)"
            )
            total_electric_consumption_series = (
                total_electric_consumption_series
                + df[matching_columns].sum(axis=1)
            )
            assigned_columns.update(matching_columns)
    ### NEGATIVE CONTRIBUTIONS
    for technology in negative_technologies:
        matching_columns = [
            column
            for column in df.columns
            if (
                technology in str(column)
                and column not in assigned_columns
            )
        ]
        if matching_columns:
            print(
                f"  - {technology}: "
                f"{len(matching_columns)} column(s)"
            )
            total_electric_consumption_series = (
                total_electric_consumption_series
                - df[matching_columns].sum(axis=1)
            )
            assigned_columns.update(matching_columns)
    ### CHECK FOR UNASSIGNED TECHNOLOGY COLUMNS
    unassigned_columns = [
        column
        for column in df.columns
        if column not in assigned_columns
        and column != "snapshot"
    ]
    if unassigned_columns:
        print("\n  WARNING: Unassigned columns:")
        for column in unassigned_columns:
            print(f"    {column}")
    ### CREATE DATAFRAME FOR THIS ZONE
    total_electric_consumption[zone] = (
        total_electric_consumption_series.to_frame(
            name="Total_Electric_Consumption"
        )
    )
    total_electric_consumption[zone].index.name = "snapshot"
    ### INSPECTION
    print(
        "  Created: Total_Electric_Consumption"
    )
    print(
        f"  Final shape: "
        f"{total_electric_consumption[zone].shape}"
    )
    print(
        "  First values:"
    )
    print(
        total_electric_consumption[zone][
            "Total_Electric_Consumption"
        ].head(2)
    )
### FINAL MESSAGE
print("\n" + "=" * 70)
print(
    "Successfully calculated "
    "'Total_Electric_Consumption' "
    "for all zones."
)
print(
    "Results stored in the new dictionary:"
    " 'total_electric_consumption'"
)
print("=" * 70)


--- Processing BE ---
  + EV charger: 1 column(s)
  + battery charger: 1 column(s)
  + gas pipeline: 8 column(s)
  + methanolisation: 1 column(s)
  + Haber-Bosch: 1 column(s)
  + H2 Electrolysis: 1 column(s)
  + H2 pipeline: 4 column(s)
  + urban central DAC: 1 column(s)
  - V2G: 1 column(s)
  - battery discharger: 1 column(s)
  - load: 2 column(s)
  Created: Total_Electric_Consumption
  Final shape: (8760, 1)
  First values:
snapshot
2013-01-01 00:00:00    4372.565747
2013-01-01 01:00:00    4917.324673
Name: Total_Electric_Consumption, dtype: float64

--- Processing FR ---
  + EV charger: 1 column(s)
  + battery charger: 1 column(s)
  + gas pipeline: 5 column(s)
  + methanolisation: 1 column(s)
  + Haber-Bosch: 1 column(s)
  + H2 Electrolysis: 1 column(s)
  + H2 pipeline: 3 column(s)
  + urban central DAC: 1 column(s)
  - V2G: 1 column(s)
  - battery discharger: 1 column(s)
  - load: 2 column(s)
  Created: Total_Electric_Consumption
  Final shape: (8760, 1)
  First values:
snapshot
2

<div style= "background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099; " >
 <span style= "font-weight: bold; font-size: 16px; " >
Section D Overview: Heat Loads, Thermal Efficiencies & Equivalencies
 </span >
 <div style= "border-top: 1px solid #000099; padding: 10px; " >
This section builds the complete thermal demand picture. It extracts heat load time series, integrates generator and link outputs (boilers, water tanks, solar thermal), and calculates the net heat balance. It then allocates these balances to individual assets using fractional contributions, applies time-varying COPs or static efficiencies, and calculates the final electric equivalency required to meet the thermal demands.<br>
        The main target of this section is calculating the <b>electric equivalent of heat consumption</b> for heater units by accounting for their <b>thermal efficiency</b> or <b>Coefficient of Performance (COP)</b>, depending on the unit type. <br>The resulting profiles represent the electricity required to produce the allocated heat output.<br>
    <b>Calculation Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each heater unit, the script identifies the <b>heat proportion</b> column and retrieves the corresponding efficiency or COP value from the metadata. The electric equivalent consumption is then calculated as:
    <div style="text-align: center; font-size: 14px;">
        <b>Electric Equivalent = Heat Proportion / Efficiency (or COP)</b>
        </div >
        This conversion ensures that the electrical input required for heat generation is correctly modeled for each unit type (e.g., resistive heaters use efficiency, heat pumps use COP).<br>
        The resulting electric equivalent profiles are stored as new columns (e.g., <b>{asset_name}_/ThermEfficiency</b>) and are ready for integration into the Dispa-SET load profiles.
        </div >
 </div >
</div >
<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
     D-01. Extracting Heat Load Time Series from CSV
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process extracts <b>heat load time series</b> from a master CSV file and organizes them into a <b>nested dictionary</b> by country and heat load category.<br> Each heat load category can have multiple associated load types (e.g., urban central heat + low-temperature heat for industry).<br>
    <b>Extraction Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each heat load category, the script maps it to one or more load string targets.<br> It then iterates through all countries, builds a list of search prefixes (primary code + acronyms), and finds columns in the master DataFrame that match both the <b>country prefix</b> and the <b>load string</b>.<br>
        Matching columns are stored in a nested dictionary: <b>country_loads_time_series[country][heat_load]</b>.
        </div>
    </div>
</div>

In [19]:
### LOAD ALL LOAD TIME SERIES FROM CSV
# _____________________________________________________________________________
### Build the input CSV path
loads_file = os.path.join(
    loads_pypsa_raw_data_folder_path,
    f"loads_{pypsa_scenario}_{data_target_year}.csv"
)
### Load the master load time-series DataFrame
master_loads_ts = pd.read_csv(loads_file, index_col=0)
print(f"Loaded load time series from:")
print(loads_file)
print(f"Shape: {master_loads_ts.shape}")
### INITIALIZE MASTER DICTIONARY
country_loads_time_series = {}
### Define all possible heat load categories to loop through
# =============================================================================
all_heat_loads = [
    'urban central heat',
    'services urban decentral heat',
    'services rural heat',
    'residential rural heat',
    'residential urban decentral heat',
]
# =============================================================================
### Initialize country keys in our master dictionary
for country in zone_names:
    country_loads_time_series[country] = {}
### EXTRACT THE REQUIRED TIME SERIES
# _____________________________________________________________________________
### Process each heat load configuration group sequentially
for current_heat_load in all_heat_loads:
    print(f"\n--- Extracting Time Series for: {current_heat_load} ---")
    ### 1. Map current heat load to its corresponding string targets
    if current_heat_load == 'urban central heat':
        loads_list = [
            'urban central heat',
            'low-temperature heat for industry'
        ]
    elif current_heat_load == 'services urban decentral heat':
        loads_list = [
            'services urban decentral heat'
        ]
    elif current_heat_load == 'services rural heat':
        loads_list = [
            'services rural heat',
            'agriculture heat'
        ]
    elif current_heat_load == 'residential rural heat':
        loads_list = [
            'residential rural heat'
        ]
    elif current_heat_load == 'residential urban decentral heat':
        loads_list = [
            'residential urban decentral heat'
        ]
    else:
        loads_list = []
    ### 2. Loop through countries
    for country in zone_names:
        ### Determine the search prefixes for this country
        ### Example: ['UK', 'GB']
        search_prefixes = [country]
        if country in zone_names_equivalences_dict:
            acronyms = zone_names_equivalences_dict[country]["Acronym"]
            clean_acronyms = [
                acr.strip()
                for acr in acronyms
                if acr.strip()
            ]
            search_prefixes.extend(clean_acronyms)
        ### 3. Find matching columns
        matching_columns = [
            col
            for col in master_loads_ts.columns
            if any(prefix in col for prefix in search_prefixes)
            and any(ld in col for ld in loads_list)
        ]
        ### 4. Extract matching time series
        if matching_columns:
            ### Keep all matching load columns side-by-side
            ts_dataframe = master_loads_ts[matching_columns].copy()
            ### Store inside nested dictionary
            country_loads_time_series[country][current_heat_load] = ts_dataframe
            print(
                f"  [{country}] Extracted "
                f"{len(matching_columns)} distinct load columns "
                f"into dictionary entry."
            )
        else:
            ### Empty DataFrame if no matching loads exist
            country_loads_time_series[country][current_heat_load] = pd.DataFrame()
            print(
                f"  [{country}] No matching load columns found."
            )
### CHECK RESULT
print(
    "\nAll time-series DataFrames have been collected into "
    "'country_loads_time_series' with separate columns!"
)
### Show dictionary structure
for country, loads in country_loads_time_series.items():
    print(country, type(loads))
    for load_name, df in loads.items():
        print("  ", load_name, type(df))
### Show the actual columns assigned to each entry
for country, loads in country_loads_time_series.items():
    print(f"\nCountry: {country}")
    for load_name, df in loads.items():
        print(f"  Load: {load_name}")
        print(f"  Columns: {df.columns.tolist()}")

Loaded load time series from:
/home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/Load_RealTime/loads_Reference_Scenario_2030.csv
Shape: (8760, 139)

--- Extracting Time Series for: urban central heat ---
  [BE] Extracted 2 distinct load columns into dictionary entry.
  [FR] Extracted 2 distinct load columns into dictionary entry.
  [DE] Extracted 2 distinct load columns into dictionary entry.
  [NL] Extracted 2 distinct load columns into dictionary entry.
  [UK] Extracted 4 distinct load columns into dictionary entry.

--- Extracting Time Series for: services urban decentral heat ---
  [BE] Extracted 1 distinct load columns into dictionary entry.
  [FR] Extracted 1 distinct load columns into dictionary entry.
  [DE] Extracted 1 distinct load columns into dictionary entry.
  [NL] Extracted 1 distinct load columns into dictionary entry.
  [UK] Extracted 2 distinct load columns into dictionary entry.

--- Extracting Time Series for: services rural heat ---
  [BE] Extracted 2 dis

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
      D-02. Adding Generator-Based Heat Profiles to Load Dictionary
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process integrates <b>generator-based heat profiles</b> (e.g., solar thermal collectors, heat loads) into the existing <b>country_loads_time_series</b> dictionary. <br>It loads a master generator time-series CSV and appends matching columns to the appropriate heat load category for each country.<br>
    <b>Integration Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each heat load category, the script maps it to corresponding generator load strings (e.g., <b>urban central solar thermal collector</b>, <b>urban central heat load</b>). It then iterates through all countries, builds search prefixes (primary code + acronyms), and finds matching columns in the <b>master_generators_ts</b> DataFrame. Matching columns are appended to the existing DataFrame for that country and heat load using <b>pd.concat()</b>.
        </div>
    </div>
</div>

In [20]:
### Build the input CSV path
loads_file = os.path.join(
    loads_pypsa_raw_data_folder_path,
    f"master_generators_ts_{pypsa_scenario}_{data_target_year}.csv"
)
### Load the master generator time-series DataFrame
master_generators_ts = pd.read_csv(loads_file, index_col=0)
print(f"Loaded generators time series from:")
print(loads_file)
print(f"Shape: {master_generators_ts.shape}")
### Master generator time-series DataFrame from the PyPSA network
### Process each heat load configuration group sequentially
for current_heat_load in all_heat_loads:
    print(f"\n--- Extracting Generator Columns for: {current_heat_load} ---")    
    ### 1. Map current heat load to its corresponding generator string targets
    if current_heat_load == 'urban central heat':
        generator_loads_list = ['urban central solar thermal collector', 'urban central heat load']
    elif current_heat_load == 'services urban decentral heat':
        generator_loads_list = ['services urban decentral solar thermal collector', 'services urban decentral heat load']
    elif current_heat_load == 'services rural heat':
        generator_loads_list = ['services rural solar thermal collector', 'services rural heat load']
    elif current_heat_load == 'residential rural heat':
        generator_loads_list = ['residential rural solar thermal collector', 'residential rural heat load']
    elif current_heat_load == 'residential urban decentral heat':
        ### Note: Keeps the slight typo 'resiential' intact just in case your PyPSA network naming uses it
        generator_loads_list = ['residential urban decentral solar thermal collector', 'residential urban decentral heat load']
    else:
        generator_loads_list = []
    ### 2. Loop through countries to find and append the generator columns
    for country in zone_names:
        ### Determine the search prefixes for this country (e.g., ['UK', 'GB'])
        search_prefixes = [country]
        if country in zone_names_equivalences_dict:
            acronyms = zone_names_equivalences_dict[country]["Acronym"]
            clean_acronyms = [acr.strip() for acr in acronyms if acr.strip()]
            search_prefixes.extend(clean_acronyms)
        ### 3. Find matching individual columns in the GENERATOR time-series matrix
        matching_gen_columns = [
            col for col in master_generators_ts.columns
            if any(prefix in col for prefix in search_prefixes) and 
               any(gen_ld in col for gen_ld in generator_loads_list)
        ]
        ### 4. If matching generator columns are found, copy them into the existing dictionary DataFrame
        if matching_gen_columns:
            ### Extract the matching generator columns
            gen_ts_dataframe = master_generators_ts[matching_gen_columns].copy()   
            ### Retrieve the already existing loads DataFrame from your dictionary
            existing_df = country_loads_time_series[country][current_heat_load]
            ### Use pd.concat along axis=1 to attach the generator columns side-by-side
            if not existing_df.empty:
                updated_df = pd.concat([existing_df, gen_ts_dataframe], axis=1)
            else:
                updated_df = gen_ts_dataframe
            ### Put the expanded DataFrame back into the dictionary
            country_loads_time_series[country][current_heat_load] = updated_df
            print(f"  [{country}] Added {len(matching_gen_columns)} generator columns to the existing collection.")
        else:
            ### No matching generators found for this group/country; leave existing data untouched
            pass
print("\nSuccessfully integrated all generator-based heat profiles into 'country_loads_time_series'!")
for country, loads in country_loads_time_series.items():
    print(country, type(loads))
    for load_name, df in loads.items():
        print("  ", load_name, type(df))
for country, loads in country_loads_time_series.items():
    print(f"\nCountry: {country}")
    for load_name, df in loads.items():
        print(f"  Load: {load_name}")
        print(f"  Columns: {df.columns.tolist()}")

Loaded generators time series from:
/home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/Load_RealTime/master_generators_ts_Reference_Scenario_2030.csv
Shape: (8760, 352)

--- Extracting Generator Columns for: urban central heat ---
  [BE] Added 2 generator columns to the existing collection.
  [FR] Added 2 generator columns to the existing collection.
  [DE] Added 2 generator columns to the existing collection.
  [NL] Added 2 generator columns to the existing collection.
  [UK] Added 4 generator columns to the existing collection.

--- Extracting Generator Columns for: services urban decentral heat ---
  [BE] Added 2 generator columns to the existing collection.
  [FR] Added 2 generator columns to the existing collection.
  [DE] Added 2 generator columns to the existing collection.
  [NL] Added 2 generator columns to the existing collection.
  [UK] Added 4 generator columns to the existing collection.

--- Extracting Generator Columns for: services rural heat ---
  [BE] Added 

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       D-03. Loading PyPSA Links Data
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process loads the <b>PyPSA links</b> CSV files for all selected zones.<br> Links represent multi-component energy system elements (e.g., CHP plants, H2 pipelines, battery chargers/dischargers, heat pumps) that connect multiple buses and have directional flows.<br>
    <b>Loading Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each country in <b>zone_names</b>, the script constructs the file path using the scenario, year, and country: <b>{base_path}/{scenario}/PowerPlants/{year}/{country}/{country}_links.csv</b>. The CSV is loaded into a DataFrame and stored in a dictionary keyed by the country code.
        </div>
    </div>
</div>

In [21]:
### LOAD PYPSA LINKS FOR ALL SELECTED ZONES
links_pypsa_dictionary = {}
for country in zone_names:
    links_file = os.path.join(
        pypsa_raw_data_folder_path,
        pypsa_scenario,
        "PowerPlants",
        data_target_year,
        country,
        f"{country}_links.csv"
    )
    print(f"\nLoading links for {country}...")
    print(f"File: {links_file}")
    links_df = pd.read_csv(links_file)
    links_pypsa_dictionary[country] = links_df
    print(
        f"{country} | "
        f"Rows: {len(links_df):,} | "
        f"Columns: {len(links_df.columns):,}"
    )
### SUMMARY
print("\n" + "=" * 80)
print("PYPSA LINKS LOADED")
print("=" * 80)
print(f"Scenario: {pypsa_scenario}")
print(f"Year:     {data_target_year}")
print(f"Zones:    {zone_names}")
print("\nLoaded DataFrames:")
for country, df in links_pypsa_dictionary.items():
    print(
        f"{country} | "
        f"Rows: {len(df):,} | "
        f"Columns: {len(df.columns):,}"
    )


Loading links for BE...
File: /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/PowerPlants/2030/BE/BE_links.csv
BE | Rows: 104 | Columns: 61

Loading links for FR...
File: /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/PowerPlants/2030/FR/FR_links.csv
FR | Rows: 120 | Columns: 61

Loading links for DE...
File: /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/PowerPlants/2030/DE/DE_links.csv
DE | Rows: 136 | Columns: 61

Loading links for NL...
File: /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/PowerPlants/2030/NL/NL_links.csv
NL | Rows: 103 | Columns: 61

Loading links for UK...
File: /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/PowerPlants/2030/UK/UK_links.csv
UK | Rows: 209 | Columns: 61

PYPSA LINKS LOADED
Scenario: Reference_Scenario
Year:     2030
Zones:    ['BE', 'FR', 'DE', 'NL', 'UK']

Loaded DataFrames:
BE | Rows: 104 | Columns: 61
FR | Rows: 120 | Columns: 61
DE | Rows: 136 | Columns: 61
NL | Rows: 103 | C

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        D-04. Loading Relevant Link Time Series
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process loads the <b>relevant link time series</b> CSV file, which contains the operational time series data for link components (e.g., CHP, H2 pipelines, EV chargers, heat pumps) in the PyPSA network.<br> These time series include flows, consumption, and generation profiles.<br>
    <b>Loading Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        The script constructs the file path: <b>{base_path}/{scenario}/Load_RealTime/relevant_links_timeseries_{scenario}_{year}.csv</b>. The CSV is loaded into a DataFrame, and a summary report displays the number of rows, columns, column names, and a preview of the first rows.
        </div>        
    </div>
</div>

In [22]:
### LOAD RELEVANT LINK TIME SERIES
relevant_links_timeseries_file = os.path.join(
    pypsa_raw_data_folder_path,
    pypsa_scenario,
    "Load_RealTime",
    f"relevant_links_timeseries_{pypsa_scenario}_{data_target_year}.csv"
)
print("Loading relevant link time series...")
print(f"File: {relevant_links_timeseries_file}")
relevant_links_timeseries_df = pd.read_csv(
    relevant_links_timeseries_file
)
print(
    f"Rows: {len(relevant_links_timeseries_df):,} | "
    f"Columns: {len(relevant_links_timeseries_df.columns):,}"
)
print("\nColumns:")
print(relevant_links_timeseries_df.columns.tolist())
print("\nFirst rows:")
print(relevant_links_timeseries_df.head())

Loading relevant link time series...
File: /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/Load_RealTime/relevant_links_timeseries_Reference_Scenario_2030.csv
Rows: 1,865,880 | Columns: 6

Columns:
['snapshot', 'link', 'country', 'heat_load', 'port', 'value']

First rows:
              snapshot                                     link country  \
0  2013-01-01 00:00:00  BE1 0 urban central water tanks charger      BE   
1  2013-01-01 01:00:00  BE1 0 urban central water tanks charger      BE   
2  2013-01-01 02:00:00  BE1 0 urban central water tanks charger      BE   
3  2013-01-01 03:00:00  BE1 0 urban central water tanks charger      BE   
4  2013-01-01 04:00:00  BE1 0 urban central water tanks charger      BE   

            heat_load port        value  
0  urban central heat   p0  2304.265994  
1  urban central heat   p0  2286.793831  
2  urban central heat   p0  2215.167589  
3  urban central heat   p0  2004.459653  
4  urban central heat   p0  1656.255785  


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        D-05. Link Technologies Associated with Each Heat-Load Group
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This dictionary defines the <b>link technologies</b> associated with each <b>heat-load group</b>.<br> It maps each heat load category to a list of link technologies that contribute to or consume heat in that sector.<br> These include boilers (gas, oil, biomass), water tank storage (chargers/dischargers), DAC, and Fischer-Tropsch processes.<br>
    <b>Mapping Logic:</b><br>
        Each heat load category is assigned a list of link technologies that are relevant to that sector:
        <div style="margin-left: 2em; font-size: 12px;">
        <li><b>Urban Central Heat</b> → gas boiler, Fischer-Tropsch, water tanks, DAC<br>
        <li><b>Services Urban Decentral Heat</b> → oil, biomass, gas boilers, water tanks<br>
        <li><b>Services Rural Heat</b> → biomass, gas, oil boilers, water tanks<br>
        <li><b>Residential Rural Heat</b> → biomass, oil, gas boilers, water tanks<br>
        <li><b>Residential Urban Decentral Heat</b> → gas, oil, biomass boilers, water tanks
    </div>
</div>

In [23]:
### LINK TECHNOLOGIES ASSOCIATED WITH EACH HEAT-LOAD GROUP
# =============================================================================
links_loads_list = {
    'urban central heat': [
        'urban central gas boiler',
        'Fischer-Tropsch',
        'urban central water tanks charger',
        'urban central water tanks discharger',
        'urban central DAC'
    ],
    'services urban decentral heat': [
        'services urban decentral oil boiler',
        'services urban decentral biomass boiler',
        'services urban decentral gas boiler',
        'services urban decentral water tanks charger',
        'services urban decentral water tanks discharger'
    ],
    'services rural heat': [
        'services rural biomass boiler',
        'services rural gas boiler',
        'services rural oil boiler',
        'services rural water tanks charger',
        'services rural water tanks discharger'
    ],
    'residential rural heat': [
        'residential rural biomass boiler',
        'residential rural oil boiler',
        'residential rural gas boiler',
        'residential rural water tanks charger',
        'residential rural water tanks discharger'
    ],
    'residential urban decentral heat': [
        'residential urban decentral gas boiler',
        'residential urban decentral oil boiler',
        'residential urban decentral biomass boiler',
        'residential urban decentral water tanks charger',
        'residential urban decentral water tanks discharger'
    ]
}

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        D-06. Adding Relevant Link Time Series to Load Dictionary
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process integrates <b>link time series</b> (e.g., boiler outputs, water tank flows, DAC, Fischer-Tropsch) into the existing <b>country_loads_time_series</b> dictionary. <br>It matches links to their corresponding heat load category by identifying the correct bus and port, then appends the time series to the existing DataFrame.<br>
    <b>Integration Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each heat load category and country, the script finds matching links from <b>links_pypsa_dictionary</b> based on technology names. For each candidate link, it identifies which bus (bus0–bus4) contains the heat load string, retrieves the corresponding time series from <b>relevant_links_timeseries_df</b>, and appends it to the existing DataFrame using <b>pd.concat(axis=1)</b>.
        </div>
    </div>
</div>

In [24]:
### ADD RELEVANT LINK TIME SERIES TO THE EXISTING COUNTRY LOAD TIME SERIES
for current_heat_load, technology_list in links_loads_list.items():
    print(
        f"\n--- Processing link outputs for: "
        f"{current_heat_load} ---"
    )
    for country in zone_names:
        ### Existing dataframe
        existing_df = country_loads_time_series[
            country
        ][
            current_heat_load
        ]
        links_df = links_pypsa_dictionary[country]
        extracted_link_columns = {}
        ### Find candidate links
        matching_links = links_df[
            links_df["name"].astype(str).apply(
                lambda name:
                    any(
                        technology in name
                        for technology in technology_list
                    )
            )
        ]
        ### Process each candidate link
        for _, link_row in matching_links.iterrows():
            link_name = link_row["name"]
            target_port = None
            ### Find which bus contains the current heat-load bus
            for bus_number in range(5):
                bus_column = f"bus{bus_number}"
                if bus_column in links_df.columns:
                    bus_value = link_row[bus_column]
                    if pd.notna(bus_value):
                        bus_name = str(bus_value)
                        if current_heat_load in bus_name:
                            target_port = f"p{bus_number}"
                            break
            ### If no heat-load bus was found, skip this link
            if target_port is None:
                continue
            ### Retrieve the corresponding saved time series
            matching_ts = relevant_links_timeseries_df[
                (relevant_links_timeseries_df["country"] == country)
                &
                (relevant_links_timeseries_df["link"] == link_name)
                &
                (relevant_links_timeseries_df["heat_load"] == current_heat_load)
                &
                (relevant_links_timeseries_df["port"] == target_port)
            ]
            if matching_ts.empty:
                continue
            ### Convert long-format time series to a Series
            link_series = (
                matching_ts
                .set_index("snapshot")["value"]
                .rename(link_name)
            )
            extracted_link_columns[link_name] = link_series
        ### Add the link time series to the EXISTING dataframe
        if extracted_link_columns:
            links_ts_dataframe = pd.DataFrame(
                extracted_link_columns
            )
            ### IMPORTANT:
            ### Keep the existing load time series and append the links.
            country_loads_time_series[
                country
            ][
                current_heat_load
            ] = pd.concat(
                [
                    existing_df,
                    links_ts_dataframe
                ],
                axis=1
            )
        ### Report result
        print(
            f"{country} | "
            f"Existing columns: {len(existing_df.columns):,} | "
            f"Link columns added: {len(extracted_link_columns):,} | "
            f"Final columns: "
            f"{len(country_loads_time_series[country][current_heat_load].columns):,}"
        )


--- Processing link outputs for: urban central heat ---
BE | Existing columns: 4 | Link columns added: 5 | Final columns: 9
FR | Existing columns: 4 | Link columns added: 5 | Final columns: 9
DE | Existing columns: 4 | Link columns added: 5 | Final columns: 9
NL | Existing columns: 4 | Link columns added: 5 | Final columns: 9
UK | Existing columns: 8 | Link columns added: 10 | Final columns: 18

--- Processing link outputs for: services urban decentral heat ---
BE | Existing columns: 3 | Link columns added: 8 | Final columns: 11
FR | Existing columns: 3 | Link columns added: 8 | Final columns: 11
DE | Existing columns: 3 | Link columns added: 8 | Final columns: 11
NL | Existing columns: 3 | Link columns added: 6 | Final columns: 9
UK | Existing columns: 6 | Link columns added: 14 | Final columns: 20

--- Processing link outputs for: services rural heat ---
BE | Existing columns: 4 | Link columns added: 8 | Final columns: 12
FR | Existing columns: 4 | Link columns added: 8 | Final colu

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        D-07. Adding Relevant Link Time Series to Load Dictionary
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process integrates <b>link time series</b> (e.g., boiler outputs, water tank flows, DAC, Fischer-Tropsch) into the existing <b>country_loads_time_series</b> dictionary. <br>It matches links to their corresponding heat load category by identifying the correct bus and port, then appends the time series to the existing DataFrame.<br>
    <b>Integration Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each heat load category and country, the script finds matching links from <b>links_pypsa_dictionary</b> based on technology names. For each candidate link, it identifies which bus (bus0–bus4) contains the heat load string, retrieves the corresponding time series from <b>relevant_links_timeseries_df</b>, and appends it to the existing DataFrame using <b>pd.concat(axis=1)</b>.
        </div>
    </div>
</div>

In [25]:
### CALCULATE NET HEAT BALANCE
for country in zone_names:
    print(
        f"\n--- Calculating Energy Balances for Country: "
        f"{country} ---"
    )
    for current_heat_load in links_loads_list:
        df = country_loads_time_series[
            country
        ][
            current_heat_load
        ]
        ### Skip empty dataframes
        if df.empty:
            print(
                f"  [{current_heat_load}] Empty dataframe - skipped."
            )
            continue
        ### Helper function
        ### Finds all columns containing the requested technology/asset name
        ### and returns the row-wise sum of their absolute values.
        def get_val(substring):
            matched_cols = [
                column
                for column in df.columns
                if substring in column
            ]
            if matched_cols:
                return (
                    df[matched_cols]
                    .abs()
                    .sum(axis=1)
                )
            return 0.0
        ### URBAN CENTRAL HEAT
        if current_heat_load == 'urban central heat':
        # =====================================================================
            net_balance = (
                get_val('urban central heat')
                + get_val('low-temperature heat for industry')
                - get_val('urban central gas boiler')
                - get_val('urban central solar thermal collector')
                - get_val('Fischer-Tropsch')
                - get_val('urban central heat load')
                + get_val('urban central water tanks charger')
                - get_val('urban central water tanks discharger')
                + get_val('urban central DAC')
            )
        # =====================================================================
        ### SERVICES URBAN DECENTRAL HEAT
        elif current_heat_load == 'services urban decentral heat':
        # =====================================================================
            net_balance = (
                get_val('services urban decentral heat')
                - get_val('services urban decentral oil boiler')
                - get_val('services urban decentral solar thermal collector')
                - get_val('services urban decentral biomass boiler')
                - get_val('services urban decentral heat load')
                - get_val('services urban decentral gas boiler')
                + get_val('services urban decentral water tanks charger')
                - get_val('services urban decentral water tanks discharger')
            )
        # =====================================================================
        ### SERVICES RURAL HEAT
        elif current_heat_load == 'services rural heat':
        # =====================================================================
            net_balance = (
                get_val('services rural heat')
                + get_val('agriculture heat')
                - get_val('services rural biomass boiler')
                - get_val('services rural solar thermal collector')
                - get_val('services rural oil boiler')
                - get_val('services rural heat load')
                - get_val('services rural gas boiler')
                + get_val('services rural water tanks charger')
                - get_val('services rural water tanks discharger')
            )
        # =====================================================================
        ### RESIDENTIAL RURAL HEAT
        elif current_heat_load == 'residential rural heat':
        # =====================================================================
            net_balance = (
                get_val('residential rural heat')
                - get_val('residential rural biomass boiler')
                - get_val('residential rural solar thermal collector')
                - get_val('residential rural oil boiler')
                - get_val('residential rural heat load')
                - get_val('residential rural gas boiler')
                + get_val('residential rural water tanks charger')
                - get_val('residential rural water tanks discharger')
            )
        # =====================================================================
        ### RESIDENTIAL URBAN DECENTRAL HEAT
        elif current_heat_load == 'residential urban decentral heat':
        # =====================================================================
            net_balance = (
                get_val('residential urban decentral heat')
                - get_val('residential urban decentral gas boiler')
                - get_val('residential urban decentral solar thermal collector')
                - get_val('residential urban decentral oil boiler')
                - get_val('residential urban decentral heat load')
                - get_val('residential urban decentral biomass boiler')
                + get_val('residential urban decentral water tanks charger')
                - get_val('residential urban decentral water tanks discharger')
            )
        # =====================================================================
        else:
            net_balance = None
        ### Add calculated balance to the EXISTING dataframe
        if net_balance is not None:
            df['net_heat_balance'] = net_balance
            print(
                f"  [{current_heat_load}] "
                f"Successfully appended 'net_heat_balance'."
            )
### COMPLETION MESSAGE
print(
    "\nAll custom equation calculations are complete!"
)
### CHECK DICTIONARY STRUCTURE
for country, loads in country_loads_time_series.items():
    print(
        f"\nCountry: {country}"
    )
    print(
        f"Dictionary type: {type(loads)}"
    )
    for load_name, df in loads.items():
        print(
            f"  Load: {load_name}"
        )
        print(
            f"  Type: {type(df)}"
        )
        print(
            f"  Rows: {len(df):,}"
        )
        print(
            f"  Columns: {len(df.columns):,}"
        )
        print(
            f"  Columns: {df.columns.tolist()}"
        )


--- Calculating Energy Balances for Country: BE ---
  [urban central heat] Successfully appended 'net_heat_balance'.
  [services urban decentral heat] Successfully appended 'net_heat_balance'.
  [services rural heat] Successfully appended 'net_heat_balance'.
  [residential rural heat] Successfully appended 'net_heat_balance'.
  [residential urban decentral heat] Successfully appended 'net_heat_balance'.

--- Calculating Energy Balances for Country: FR ---
  [urban central heat] Successfully appended 'net_heat_balance'.
  [services urban decentral heat] Successfully appended 'net_heat_balance'.
  [services rural heat] Successfully appended 'net_heat_balance'.
  [residential rural heat] Successfully appended 'net_heat_balance'.
  [residential urban decentral heat] Successfully appended 'net_heat_balance'.

--- Calculating Energy Balances for Country: DE ---
  [urban central heat] Successfully appended 'net_heat_balance'.
  [services urban decentral heat] Successfully appended 'net_heat_

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        D-08. Allocating Net Heat Balances Using Fractions from Power-Plant CSV
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process allocates the <b>net heat balance</b> (net_heat_balance) across individual power-plant assets using their <b>fractional contributions</b> from the PyPSA power-plant CSV files. Each asset receives a proportional share of the net heat balance based on its fraction value.<br>
    <b>Allocation Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each country and heat load category, the script loads the power-plant DataFrame and checks for the existence of the fraction column (e.g., <b>urban central heat_fraction</b>). It then selects assets with a positive fraction and creates a new column for each asset: <b>{asset_name}_proportion</b> = net_heat_balance × fraction_value. These allocated columns are appended to the existing time-series DataFrame.
        </div>
    </div>
</div>

In [26]:
### ALLOCATE NET HEAT BALANCES USING FRACTIONS FROM PYPSA POWER-PLANT CSV FILES
for country in zone_names:
    print(f"\n--- Allocating Net Balances via Fractions for Country: {country} ---")
    ### Build the path to the country's power-plant CSV
    power_plants_file = os.path.join(
        pypsa_raw_data_folder_path,
        pypsa_scenario,
        "PowerPlants",
        data_target_year,
        country,
        f"pypsa_power_plants_{country}_{data_target_year}.csv"
    )
    print(f"  Loading power-plant data:")
    print(f"  {power_plants_file}")
    ### Load the country's power-plant dataframe
    master_df = pd.read_csv(power_plants_file)
    ### Use 'name' as the asset identifier
    master_df = master_df.set_index("name")
    ### Process every heat-load category
    for current_heat_load in all_heat_loads:
        ### Example:
        ### 'urban central heat' ->
        ### 'urban central heat_fraction'
        fraction_col = f"{current_heat_load}_fraction"
        ### Check whether the fraction column exists
        if fraction_col not in master_df.columns:
            print(
                f"  [{current_heat_load}] "
                f"Fraction column '{fraction_col}' not found. Skipping."
            )
            continue
        ### Retrieve the existing time-series dataframe
        ts_df = country_loads_time_series[country][current_heat_load]
        ### Check whether net_heat_balance exists
        if ts_df.empty or "net_heat_balance" not in ts_df.columns:
            print(
                f"  [{current_heat_load}] "
                f"'net_heat_balance' not available. Skipping."
            )
            continue
        ### Select assets with a positive fraction
        active_assets = master_df[
            master_df[fraction_col] > 0
        ]
        allocated_columns = {}
        ### Allocate the net heat balance to each active asset
        for asset_name, row in active_assets.iterrows():
            fraction_value = float(row[fraction_col])
            ### New column name
            new_column_header = f"{asset_name}_proportion"
            ### Apply the fraction to the net heat balance
            allocated_columns[new_column_header] = (
                ts_df["net_heat_balance"] * fraction_value
            )
        ### Append the allocated profiles to the existing dataframe
        if allocated_columns:
            allocated_df = pd.DataFrame(
                allocated_columns,
                index=ts_df.index
            )
            country_loads_time_series[country][current_heat_load] = pd.concat(
                [
                    ts_df,
                    allocated_df
                ],
                axis=1
            )
            print(
                f"  [{current_heat_load}] "
                f"Distributed net balance across "
                f"{len(allocated_columns)} assets."
            )
### SUMMARY
print("\nAll fractional asset proportion columns successfully generated!")
for country, loads in country_loads_time_series.items():
    print(country, type(loads))
    for load_name, df in loads.items():
        print(
            "  ",
            load_name,
            type(df)
        )
for country, loads in country_loads_time_series.items():
    print(f"\nCountry: {country}")
    for load_name, df in loads.items():
        print(f"  Load: {load_name}")
        print(f"  Columns: {df.columns.tolist()}")


--- Allocating Net Balances via Fractions for Country: BE ---
  Loading power-plant data:
  /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/PowerPlants/2030/BE/pypsa_power_plants_BE_2030.csv
  [urban central heat] Distributed net balance across 7 assets.
  [services urban decentral heat] Distributed net balance across 4 assets.
  [services rural heat] Distributed net balance across 5 assets.
  [residential rural heat] Distributed net balance across 5 assets.
  [residential urban decentral heat] Distributed net balance across 4 assets.

--- Allocating Net Balances via Fractions for Country: FR ---
  Loading power-plant data:
  /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/PowerPlants/2030/FR/pypsa_power_plants_FR_2030.csv
  [urban central heat] Distributed net balance across 7 assets.
  [services urban decentral heat] Distributed net balance across 6 assets.
  [services rural heat] Distributed net balance across 7 assets.
  [residential rural heat] Distribut

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        D-09. Loading COP Time Series and Calculating Thermal Efficiencies for Heat Load Assets
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process loads the <b>COP (Coefficient of Performance) time series</b> for heat pump units and calculates the <b>thermal efficiency-adjusted</b> profiles for each heat load asset. It supports both <b>time-varying COP</b> (for heat pumps) and <b>static efficiency</b> (for resistive heaters and boilers).<br>
    <b>Calculation Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each asset, the script first checks whether a <b>COP time series</b> exists in the loaded COP DataFrame. If available, it uses the time-varying COP to calculate the electric equivalent:_/
                        <div style="text-align: center;">
            <b>ThermEfficiency = proportion / COP</b>
                        </div>
        </div>
            If no COP series is found, it falls back to the <b>static efficiency</b> from the power-plant metadata, usin the bus-to-efficiency mapping to find the correct efficiency value.
    </div>
</div>

In [27]:
### LOAD COP TIME SERIES
cop_file = os.path.join(
    loads_pypsa_raw_data_folder_path,
    f"COP_ts_{pypsa_scenario}_{data_target_year}.csv"
)
print("\n--- Loading COP Time Series ---")
print(f"File: {cop_file}")
cop_ts_df = pd.read_csv(cop_file)
### Convert snapshot to datetime
cop_ts_df["snapshot"] = pd.to_datetime(cop_ts_df["snapshot"])
### Use snapshot as index
cop_ts_df = cop_ts_df.set_index("snapshot")
print(
    f"COP time series loaded: "
    f"{len(cop_ts_df):,} snapshots | "
    f"{len(cop_ts_df.columns):,} COP series"
)
print("COP columns:")
for col in cop_ts_df.columns:
    print(f"  {col}")
### BUS -> EFFICIENCY COLUMN MAPPING
# =============================================================================
bus_to_eff_map = {
    'bus':  'efficiency',
    'bus0': 'efficiency',
    'bus1': 'efficiency',
    'bus2': 'efficiency2',
    'bus3': 'efficiency3',
    'bus4': 'efficiency4'
}
# =============================================================================
### CALCULATE THERMAL EFFICIENCIES
for country in zone_names:
    print(f"\n--- Calculating Thermal Efficiencies for Country: {country} ---")
    ### Load power-plant metadata
    power_plants_file = os.path.join(
        pypsa_raw_data_folder_path,
        pypsa_scenario,
        "PowerPlants",
        data_target_year,
        country,
        f"pypsa_power_plants_{country}_{data_target_year}.csv"
    )
    print(f"  Loading metadata from:")
    print(f"  {power_plants_file}")
    master_df = pd.read_csv(power_plants_file)
    master_df = master_df.set_index("name")
    ### Process each heat-load category
    for current_heat_load, ts_df in country_loads_time_series[country].items():
        if ts_df.empty:
            continue
        ### Identify proportion columns
        proportion_cols = [
            col
            for col in ts_df.columns
            if col.endswith("_proportion")
        ]

        if not proportion_cols:
            continue
        calculated_efficiencies = {}
        ### PROCESS EACH ASSET
        for col in proportion_cols:
            ### Remove "_proportion"
            asset_name = col[:-11]
            ### Check that asset exists in power-plant metadata
            if asset_name not in master_df.index:
                print(
                    f"  [{current_heat_load}] "
                    f"Asset '{asset_name}' not found in metadata. Skipping."
                )
                continue
            asset_row = master_df.loc[asset_name]
            ### FIRST: CHECK WHETHER THIS ASSET HAS A COP TIME SERIES
            if asset_name in cop_ts_df.columns:
                print(
                    f"  [{current_heat_load}] "
                    f"Using COP time series for '{asset_name}'."
                )
                cop_series = cop_ts_df[asset_name].reindex(ts_df.index)
                calculated_efficiencies[
                    f"{asset_name}_/ThermEfficiency"
                ] = ts_df[col] / cop_series
                continue
            ### OTHERWISE: USE STATIC THERMAL EFFICIENCY
            matched_eff_col = None
            for bus_col, eff_col in bus_to_eff_map.items():
                if bus_col not in asset_row:
                    continue
                bus_value = asset_row[bus_col]
                if pd.isna(bus_value):
                    continue
                if current_heat_load in str(bus_value):
                    matched_eff_col = eff_col
                    break
            ### No matching bus
            if matched_eff_col is None:
                print(
                    f"  [{current_heat_load}] "
                    f"No matching bus found for '{asset_name}'."
                )
                continue
            ### Obtain static efficiency
            divisor = None
            if matched_eff_col in asset_row:
                efficiency_value = asset_row[matched_eff_col]
                if pd.notna(efficiency_value):
                    efficiency_value = float(efficiency_value)
                    if efficiency_value != 0:
                        divisor = efficiency_value
            ### Calculate using static efficiency
            if divisor is not None:
                new_col = f"{asset_name}_/ThermEfficiency"
                calculated_efficiencies[new_col] = (
                    ts_df[col] / divisor
                )
            else:
                print(
                    f"  [{current_heat_load}] "
                    f"Could not find a valid efficiency for "
                    f"'{asset_name}'."
                )
        ### ADD CALCULATED COLUMNS
        if calculated_efficiencies:
            eff_df = pd.DataFrame(
                calculated_efficiencies,
                index=ts_df.index
            )
            country_loads_time_series[country][current_heat_load] = pd.concat(
                [
                    ts_df,
                    eff_df
                ],
                axis=1
            )

            print(
                f"  [{current_heat_load}] "
                f"Added {len(calculated_efficiencies)} "
                f"'_/ThermEfficiency' columns."
            )
print("\nThermal efficiency calculation completed successfully!")


--- Loading COP Time Series ---
File: /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/Load_RealTime/COP_ts_Reference_Scenario_2030.csv
COP time series loaded: 8,760 snapshots | 70 COP series
COP columns:
  BE1 0 residential rural air heat pump-2030
  BE1 0 residential rural ground heat pump-2030
  BE1 0 residential urban decentral air heat pump-2030
  BE1 0 services rural air heat pump-2030
  BE1 0 services rural ground heat pump-2030
  BE1 0 services urban decentral air heat pump-2030
  BE1 0 urban central air heat pump-2030
  DE1 0 residential rural air heat pump-2030
  DE1 0 residential rural ground heat pump-2015
  DE1 0 residential rural ground heat pump-2019
  DE1 0 residential rural ground heat pump-2030
  DE1 0 residential urban decentral air heat pump-2015
  DE1 0 residential urban decentral air heat pump-2019
  DE1 0 residential urban decentral air heat pump-2030
  DE1 0 services rural air heat pump-2030
  DE1 0 services rural ground heat pump-2015
  DE1 0 servi

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        D-10. Extracting Heater Consumption Electric Equivalency (with VPP-Based Filtering)
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process extracts all <b>_/ThermEfficiency</b> columns from the <b>country_loads_time_series</b> dictionary and consolidates them into a new dictionary (<b>heaters_consumption_elect_equivalency</b>). These columns represent the <b>electric equivalency of heater consumption</b> (fuel input demand) for various technologies.<br> The extraction is <b>conditionally filtered based on VPP mode</b> to exclude resistive heaters when VPP is enabled.<br>
    <b>Extraction Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each country, the script iterates through all heat load categories and identifies columns containing <b>_/ThermEfficiency</b> in their names. The selection is filtered based on <b>VPP</b> mode:
                <div style="margin-left: 2em;">
        <b>When VPP = True</b> → Excludes columns containing <b>"resistive"</b> (resistive heaters are handled separately by VPP logic)<br>
        <b>When VPP = False</b> → Includes all <b>_/ThermEfficiency</b> columns
                </div>
        The selected columns are concatenated <b>horizontally</b> (along the column axis) into a single DataFrame per country.
        </div>
    </div>
</div>

In [28]:
heaters_consumption_elect_equivalency = {}
for country, heat_loads_dict in country_loads_time_series.items():
    country_efficiency_dfs = []
    for heat_category, df in heat_loads_dict.items():
        # SELECT /ThermEfficiency COLUMNS ACCORDING TO VPP FLAG
        if VPP:
            # VPP activated:
            # Exclude all resistive heater columns
            efficiency_cols = [
                col
                for col in df.columns
                if (
                    "_/ThermEfficiency" in str(col)
                    and "resistive" not in str(col).lower()
                )
            ]
        else:
            # VPP deactivated:
            # Include all /ThermEfficiency columns
            efficiency_cols = [
                col
                for col in df.columns
                if "_/ThermEfficiency" in str(col)
            ]
        # STORE SELECTED COLUMNS
        if efficiency_cols:
            country_efficiency_dfs.append(
                df[efficiency_cols]
            )
    # COMBINE ALL HEAT CATEGORIES FOR THE COUNTRY
    if country_efficiency_dfs:
        heaters_consumption_elect_equivalency[country] = pd.concat(
            country_efficiency_dfs,
            axis=1
        )
    else:
        heaters_consumption_elect_equivalency[country] = pd.DataFrame(
            index=next(iter(heat_loads_dict.values())).index
        )
# FINAL INFORMATION
print("\nProcess completed.")
for country, df in heaters_consumption_elect_equivalency.items():
    print(
        f" {country}: "
        f"{df.shape[1]} efficiency columns extracted"
    )


Process completed.
 BE: 25 efficiency columns extracted
 FR: 33 efficiency columns extracted
 DE: 25 efficiency columns extracted
 NL: 27 efficiency columns extracted
 UK: 53 efficiency columns extracted


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        D-11. Summing All Heater Consumption Electric Equivalency Columns
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process aggregates all heater consumption electric equivalency columns for each country into a <b>single total time series</b>. The resulting DataFrame contains one column per country (<b>total_heaters_consumption_elect_equivalency</b>) representing the <b>total electric equivalency of all heater consumption</b> at each snapshot.<br>
    <b>Aggregation Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each country, the script sums all columns of the DataFrame <b>horizontally</b> (along the column axis) using <b>df.sum(axis=1)</b>. This produces a single Series where each value is the sum of all heater electric equivalency components at that snapshot.
        <br>The Series is then converted to a DataFrame with a single column named <b>"total_heaters_consumption_elect_equivalency"</b>, preserving the snapshot as the index.
        </div>
        The resulting dictionary, <b>total_heaters_consumption_elect_equivalency</b>, contains one DataFrame per country with a single column representing the total electric equivalency of all heater consumption. This aggregated profile is ready for Dispa-SET load formatting.
    </div>
</div>

In [29]:
total_heaters_consumption_elect_equivalency = {}
for country, df in heaters_consumption_elect_equivalency.items():
    total_consumption = df.sum(axis=1)
    total_heaters_consumption_elect_equivalency[country] = (
        total_consumption.to_frame(
            name="total_heaters_consumption_elect_equivalency"
        )
    )
    total_heaters_consumption_elect_equivalency[country].index.name = "snapshot"
print("\nProcess completed.")
for country, df in total_heaters_consumption_elect_equivalency.items():
    print(
        f"{country}: {len(df)} snapshots processed, "
        f"column = '{df.columns[0]}'"
    )


Process completed.
BE: 8760 snapshots processed, column = 'total_heaters_consumption_elect_equivalency'
FR: 8760 snapshots processed, column = 'total_heaters_consumption_elect_equivalency'
DE: 8760 snapshots processed, column = 'total_heaters_consumption_elect_equivalency'
NL: 8760 snapshots processed, column = 'total_heaters_consumption_elect_equivalency'
UK: 8760 snapshots processed, column = 'total_heaters_consumption_elect_equivalency'


<div style= "background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099; " >
 <span style= "font-weight: bold; font-size: 16px; " >
Section E Overview: Sector Power Load Calculation
 </span >
 <div style= "border-top: 1px solid #000099; padding: 10px; " >
This section calculates the baseline Sector Power Load for each country.<br> It combines three core components: the total baseline electric load, the total electric consumption from sector-coupled links, and the total electric equivalency of heater consumptions, resulting in a unified power sector demand profile.
 </div >
 </div >
<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        E-1. Calculating Sector Power Load
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process calculates the <b>Sector Power Load</b> for each country by combining three components: <b>total electric load</b>, <b>total electric consumption</b>, and <b>total heaters consumption electric equivalency</b>. <br>The result is a single time series representing the total sector power load for each country.<br>
    <b>Calculation Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each country, the script retrieves three DataFrames:
            <div style="margin-left: 2em;">
        <b>1.</b> <b>total_electric_load</b> from <code>total_electric_loads</code><br>
        <b>2.</b> <b>Total_Electric_Consumption</b> from <code>total_electric_consumption</code><br>
        <b>3.</b> <b>total_heaters_consumption_elect_equivalency</b> from <code>total_heaters_consumption_elect_equivalency</code>
            </div>
        All indices are converted to DatetimeIndex and the three columns are combined using <b>pd.concat(axis=1, join="inner")</b>. The Sector Power Load is then calculated as:
            <div style="text-align: center; font-size: 14px;">
        <b>Sector_Power_Load = total_electric_load + Total_Electric_Consumption + total_heaters_consumption_elect_equivalency</b>
            </div>
        The resulting DataFrame is stored in the <b>Sector_Power_Load</b> dictionary with a single column named "Sector_Power_Load" and a snapshot index.
        <br>A summary report displays the shape and first values per country.
        </div>
    </div>
</div>

In [30]:
### CALCULATE SECTOR POWER LOAD
Sector_Power_Load = {}
### PROCESS EACH COUNTRY
for country in total_electric_loads.keys():
    print(f"\n--- Processing {country} ---")
    # Get the three source DataFrames
    electric_loads_df = total_electric_loads[country].copy()
    electric_consumption_df = (
        total_electric_consumption[country].copy()
    )
    heaters_consumption_df = (
        total_heaters_consumption_elect_equivalency[country].copy()
    )
    # Make sure all indexes are DatetimeIndex
    electric_loads_df.index = pd.to_datetime(
        electric_loads_df.index
    )
    electric_consumption_df.index = pd.to_datetime(
        electric_consumption_df.index
    )
    heaters_consumption_df.index = pd.to_datetime(
        heaters_consumption_df.index
    )
    # Select the relevant columns
    electric_load = electric_loads_df[
        "total_electric_load"
    ]
    electric_consumption = electric_consumption_df[
        "Total_Electric_Consumption"
    ]
    heaters_consumption = (
        heaters_consumption_df[
            "total_heaters_consumption_elect_equivalency"
        ]
    )
    # Combine using the common snapshot index
    combined_df = pd.concat(
        [
            electric_load.rename("total_electric_load"),
            electric_consumption.rename(
                "Total_Electric_Consumption"
            ),
            heaters_consumption.rename(
                "total_heaters_consumption_elect_equivalency"
            ),
        ],
        axis=1,
        join="inner"
    )
    # Calculate Sector Power Load
    combined_df["Sector_Power_Load"] = (
        combined_df["total_electric_load"]
        + combined_df["Total_Electric_Consumption"]
        + combined_df[
            "total_heaters_consumption_elect_equivalency"
        ]
    )
    # Keep only the final result
    Sector_Power_Load[country] = combined_df[
        ["Sector_Power_Load"]
    ].copy()
    Sector_Power_Load[country].index.name = "snapshot"
    # Inspection
    print(
        f"  Created: Sector_Power_Load"
    )
    print(
        f"  Shape: "
        f"{Sector_Power_Load[country].shape}"
    )
    print(
        "  First values:"
    )
    print(
        Sector_Power_Load[country].head(2)
    )
### FINAL MESSAGE
print("\n" + "=" * 70)
print(
    "Successfully calculated "
    "'Sector_Power_Load' "
    "for all countries."
)
print("=" * 70)


--- Processing BE ---
  Created: Sector_Power_Load
  Shape: (8760, 1)
  First values:
                     Sector_Power_Load
snapshot                              
2013-01-01 00:00:00       22961.222891
2013-01-01 01:00:00       23364.638164

--- Processing FR ---
  Created: Sector_Power_Load
  Shape: (8760, 1)
  First values:
                     Sector_Power_Load
snapshot                              
2013-01-01 00:00:00      104519.036188
2013-01-01 01:00:00      104019.803187

--- Processing DE ---
  Created: Sector_Power_Load
  Shape: (8760, 1)
  First values:
                     Sector_Power_Load
snapshot                              
2013-01-01 00:00:00       98591.415053
2013-01-01 01:00:00       97134.133069

--- Processing NL ---
  Created: Sector_Power_Load
  Shape: (8760, 1)
  First values:
                     Sector_Power_Load
snapshot                              
2013-01-01 00:00:00       25936.857853
2013-01-01 01:00:00       25355.925102

--- Processing UK ---
  Cre

<div style= "background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099; " >
 <span style= "font-weight: bold; font-size: 16px; " >
Section F Overview: DHW & Space Heating Split, Reshaping & Conversion
 </span >
 <div style= "border-top: 1px solid #000099; padding: 10px; " >
This section isolates the resistive heater demands for VPP modeling.<br> It splits the total resistive load into Domestic Hot Water (DHW) and Space Heating using a seasonal-baseload approach, reshapes the DHW profile using historical 2023 reference profiles to ensure realistic diurnal patterns, and converts both thermal demands back into electrical demands using the average efficiency of the resistive heaters.
 </div >
 </div >
<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
      F-1. Seasonal Split of Resistive Heating into DHW and Space Heating
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process splits resistive heater load profiles into <b>Domestic Hot Water (DHW)</b> and <b>Space Heating</b> components using a <b>seasonal-baseload approach</b>.<br> The split is based on the average resistive load during summer months, which is used to estimate the DHW baseline, while space heating is calculated as the residual after applying a representative diurnal profile.<br>
    <b>Splitting Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each node, the script aggregates all resistive heater proportion columns and performs the following steps:
            <div style="margin-left: 2em;">
        <b>1.</b> <b>DHW Baseline:</b> Calculates the average resistive load during summer months (June, July, August) as the initial DHW baseline.<br>
        <b>2.</b> <b>Diurnal Profile:</b> Estimates a representative 24-hour diurnal profile from the hourly means of the load data.<br>
        <b>3.</b> <b>DHW Profile:</b> Applies the diurnal profile to the DHW baseline and smooths it over 24 hours.<br>
        <b>4.</b> <b>Energy Preservation:</b> Scales the DHW profile to preserve the annual DHW energy.<br>
        <b>5.</b> <b>Space Heating:</b> Calculates space heating as the residual: <b>Space Heating = Total Resistive - DHW</b>.
            </div>
        The resulting DHW and space heating demand profiles are added as new columns (<b>{node} DHW_demand</b> and <b>{node} SpaceHeating_demand</b>) for each processed node, enabling detailed thermal load modeling in Dispa-SET.
        </div>
    </div>
</div>

In [31]:
### HELPER FUNCTION: SEASONAL SPLIT OF RESISTIVE HEATING
def split_heat_demand_seasonal(
    heat_series,
    summer_months=[6, 7, 8],
    smoothing_window_hours=24,
    time_step=data_target_time_step
):
    """
    Split a resistive-heater load profile into:
        1. DHW demand
        2. Space-heating demand
    The split is based on a seasonal-baseload approach:
    - The average resistive load during summer is used as the initial DHW
      baseline.
    - A representative diurnal profile is applied.
    - The DHW profile is smoothed over 24 hours.
    - Space heating is obtained as the residual.
    Parameters
    ----------
    heat_series : pandas.Series
        Total resistive-heater load for one node.
    summer_months : list
        Months considered representative of the summer DHW baseline.
    smoothing_window_hours : int
        Smoothing window expressed in hours, independent of the model
        temporal resolution.
    time_step : str
        Model temporal resolution. Expected values:
        '1h', '30min', '15min'.
    Returns
    -------
    dhw : pandas.Series
        Estimated domestic hot-water demand.
    sh : pandas.Series
        Estimated space-heating demand.
    """
    heat = heat_series.copy()
    ### 1. Ensure numeric values
    heat = pd.to_numeric(heat, errors="coerce").fillna(0.0)
    ### 2. Ensure datetime index
    if not isinstance(heat.index, pd.DatetimeIndex):
        heat.index = pd.to_datetime(heat.index)
    ### 3. Determine number of observations corresponding to 24 hours
    time_step_to_minutes = {
        "1h": 60,
        "60min": 60,
        "30min": 30,
        "15min": 15
    }
    if time_step not in time_step_to_minutes:
        raise ValueError(
            f"Unsupported time step '{time_step}'. "
            f"Expected one of: {list(time_step_to_minutes.keys())}"
        )
    observations_per_hour = 60 / time_step_to_minutes[time_step]
    smoothing_window = max(
        1,
        int(round(smoothing_window_hours * observations_per_hour))
    )
    ### 4. Estimate DHW baseline from summer months
    summer_mask = heat.index.month.isin(summer_months)
    if summer_mask.any():
        dhw_baseline = heat.loc[summer_mask].mean()
    else:
        dhw_baseline = heat.mean()
    ### If there is no resistive load, return zeros
    if dhw_baseline <= 0:
        dhw = pd.Series(0.0, index=heat.index)
        sh = pd.Series(0.0, index=heat.index)
        return dhw, sh
    ### 5. Estimate representative diurnal profile
    hourly_mean = heat.groupby(heat.index.hour).mean()
    mean_of_hourly = hourly_mean.mean()
    if mean_of_hourly > 0:
        hourly_profile = hourly_mean / mean_of_hourly
        ### Map each timestamp to its corresponding hour-of-day profile
        diurnal_factor = pd.Series(
            heat.index.hour.map(hourly_profile),
            index=heat.index
        )
        dhw = dhw_baseline * diurnal_factor
    else:
        dhw = pd.Series(
            dhw_baseline,
            index=heat.index
        )
    ### 6. Smooth DHW profile
    dhw = (
        dhw
        .rolling(
            window=smoothing_window,
            center=True,
            min_periods=1
        )
        .mean()
    )
    ### 7. Preserve the intended annual DHW energy
    target_energy = dhw_baseline * len(dhw)
    current_dhw_energy = dhw.sum()
    if current_dhw_energy > 0:
        dhw *= target_energy / current_dhw_energy
    ### 8. Space heating = residual
    sh = heat - dhw
    ### Numerical / physical correction
    sh = sh.clip(lower=0.0)
    ### 9. Ensure exact decomposition of the original resistive load
    dhw = heat - sh
    return dhw, sh
### IN-PLACE SEASONAL SPLIT OF RESISTIVE-HEATER PROPORTIONS
for country_code, heat_dict in country_loads_time_series.items():
    for heat_category in all_heat_loads:
        ### Check that the heat category exists
        if heat_category not in heat_dict:
            continue
        df = heat_dict[heat_category]
        if df.empty:
            continue
        ### 1. Identify resistive-heater proportion columns
        ### These columns were created by the previous code, for example:
        ### BE0 0 residential urban decentral resistive heater_proportion
        ### BE0 1 residential urban decentral resistive heater_proportion
        matching_cols = [
            col
            for col in df.columns
            if (
                "resistive heater" in str(col).lower()
                and "_proportion" in str(col).lower()
            )
        ]
        if not matching_cols:
            continue
        ### 2. Extract node prefixes
        ### Examples:
        ###   BE0 0
        ###   BE0 1
        ###   BE0 2
        ###   FR0 0
        ### The node is assumed to be the first country/network identifier
        ### before the technology description.
        nodes_in_df = set()
        for col in matching_cols:
            col_string = str(col)
            node_match = re.match(
                r"^([A-Z]{2}\d+\s+\d+)",
                col_string
            )
            if node_match:
                nodes_in_df.add(
                    node_match.group(1).strip()
                )
        ### Fallback:
        ### If no explicit node can be extracted, use the country code.
        if not nodes_in_df:
            nodes_in_df = {country_code}
        ### 3. Process each node independently
        processed_nodes = []
        for node in sorted(nodes_in_df):
            ### Select only resistive-heater proportion columns belonging
            ### to this node.
            if node == country_code and len(nodes_in_df) == 1:
                node_cols = matching_cols
            else:
                node_cols = [
                    col
                    for col in matching_cols
                    if str(col).startswith(node)
                ]
            if not node_cols:
                continue
            ### 4. Aggregate all resistive-heater proportions for this node
            total_resistive = (
                df[node_cols]
                .sum(axis=1)
                .fillna(0.0)
            )
            ### 5. Perform the seasonal DHW / space-heating split
            if total_resistive.sum() > 0:
                dhw, space_heating = split_heat_demand_seasonal(
                    total_resistive,
                    time_step=data_target_time_step
                )
            else:
                dhw = pd.Series(
                    0.0,
                    index=df.index
                )
                space_heating = pd.Series(
                    0.0,
                    index=df.index
                )
            ### 6. Add node-prefixed results
            df[f"{node} total_resistive_load"] = total_resistive
            df[f"{node} DHW_demand"] = dhw
            df[f"{node} SpaceHeating_demand"] = space_heating
            processed_nodes.append(node)
        ### 7. Report results
        if processed_nodes:
            print(
                f"✅ [{country_code}] [{heat_category}] "
                f"Added seasonal split for nodes: "
                f"{', '.join(processed_nodes)}"
            )
        else:
            print(
                f"⚠️ [{country_code}] [{heat_category}] "
                f"No nodes could be processed."
            )
print(
    "\nDone! Seasonal DHW and SpaceHeating demand "
    "have been added to 'country_loads_time_series'."
)

✅ [BE] [urban central heat] Added seasonal split for nodes: BE1 0
✅ [BE] [services urban decentral heat] Added seasonal split for nodes: BE1 0
✅ [BE] [services rural heat] Added seasonal split for nodes: BE1 0
✅ [BE] [residential rural heat] Added seasonal split for nodes: BE1 0
✅ [BE] [residential urban decentral heat] Added seasonal split for nodes: BE1 0
✅ [FR] [urban central heat] Added seasonal split for nodes: FR1 0
✅ [FR] [services urban decentral heat] Added seasonal split for nodes: FR1 0
✅ [FR] [services rural heat] Added seasonal split for nodes: FR1 0
✅ [FR] [residential rural heat] Added seasonal split for nodes: FR1 0
✅ [FR] [residential urban decentral heat] Added seasonal split for nodes: FR1 0
✅ [DE] [urban central heat] Added seasonal split for nodes: DE1 0
✅ [DE] [services urban decentral heat] Added seasonal split for nodes: DE1 0
✅ [DE] [services rural heat] Added seasonal split for nodes: DE1 0
✅ [DE] [residential rural heat] Added seasonal split for nodes: DE1 0


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        F-2. Reshaping DHW Profiles Using a Reference Profile
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process reshapes the <b>Domestic Hot Water (DHW) demand profiles</b> for each node using a <b>country-specific reference profile</b> (typically a 2023 historical profile). <br>The reshaping preserves the <b>total annual DHW energy</b> while imposing the <b>temporal shape</b> of the reference profile, ensuring that the resulting DHW demand follows realistic diurnal and seasonal patterns.<br>
    <b>Reshaping Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each country and node, the script loads the country-specific reference DHW profile (e.g., <b>Database/Boundary_Sector/BoundarySectorDemand/BE/2023.csv</b>).
        <br>
        It then applies the <b>reshape_dhw_series</b> function, which:
            <div style="margin-left: 2em;">
        <b>1.</b> Extracts the total annual DHW energy from the original profile<br>
        <b>2.</b> Maps the reference profile onto the target timestamps using <b>month + day + hour + minute</b> keys<br>
        <b>3.</b> Normalizes the mapped reference profile to sum to 1<br>
        <b>4.</b> Rescales the normalized shape to match the original total DHW energy<br>
        <b>5.</b> Handles missing timestamps (e.g., February 29 in leap years) by filling with the mean reference value
            </div>
        The resulting reshaped DHW profiles are added as new columns (<b>{node} DHW_demand_reshaped</b>), while the original DHW columns are preserved. An energy balance check confirms that the total annual DHW energy is maintained.
        </div>
    </div>
</div>

In [32]:
### USER SETTINGS & PATH CONFIGURATION
root_dir = dispaSET_unleash_folder_path.joinpath("Database")
### HELPER FUNCTION: RESHAPE DHW PROFILE USING A REFERENCE PROFILE
### ENERGY-PRESERVING AND TIME-STEP AWARE
def reshape_dhw_series(dhw_series, reference_series):
    """
    Reshape a DHW demand series using the temporal shape of a reference
    profile while preserving the total DHW energy of the original series.
    Parameters
    ----------
    dhw_series : pandas.Series
        Target DHW demand series to be reshaped.
    reference_series : pandas.Series
        Reference DHW profile, normally the country 2023 profile.
    Returns
    -------
    pandas.Series
        Reshaped DHW profile with the same index as dhw_series and exactly
        the same total energy as the original DHW series.
    """
    ### 1. Copy and clean target DHW series
    dhw = pd.to_numeric(
        dhw_series,
        errors="coerce"
    ).fillna(0.0)
    ### Make sure the index is datetime
    if not isinstance(dhw.index, pd.DatetimeIndex):
        dhw.index = pd.to_datetime(dhw.index)
    ### Remove timezone if present
    if dhw.index.tz is not None:
        dhw.index = dhw.index.tz_localize(None)
    ### Total DHW energy to preserve
    target_energy = dhw.sum()
    ### If there is no DHW energy, return zeros
    if target_energy <= 0:
        return pd.Series(
            0.0,
            index=dhw.index
        )
    ### 2. Clean reference profile
    reference = pd.to_numeric(
        reference_series,
        errors="coerce"
    ).fillna(0.0)
    ### Make sure reference index is datetime
    if not isinstance(reference.index, pd.DatetimeIndex):
        reference.index = pd.to_datetime(reference.index)
    ### Remove timezone if present
    if reference.index.tz is not None:
        reference.index = reference.index.tz_localize(None)
    ### Remove negative values
    reference = reference.clip(lower=0.0)
    if reference.sum() <= 0:
        print(
            "   ⚠️ Reference profile has zero energy. "
            "Original DHW profile will be retained."
        )
        return dhw.copy()
    ### 3. Convert reference profile to the same temporal resolution
    ### Determine target temporal resolution from the target index.
    ### This assumes a regular time series, which is the case for the
    ### PyPSA / Dispa-SET profiles being processed here.
    if len(dhw.index) > 1:
        target_delta = (
            dhw.index[1:] - dhw.index[:-1]
        ).median()
        target_minutes = target_delta.total_seconds() / 60
    else:
        target_minutes = None
    ### If the reference and target have different numbers of observations,
    ### interpolate the reference profile to the target temporal resolution.
    if target_minutes is not None and len(reference.index) > 1:
        reference_delta = (
            reference.index[1:] - reference.index[:-1]
        ).median()
        reference_minutes = (
            reference_delta.total_seconds() / 60
        )
        if reference_minutes != target_minutes:
            reference = (
                reference
                .resample(
                    f"{int(target_minutes)}min"
                )
                .mean()
                .interpolate(method="time")
            )
    ### 4. Build an annual temporal shape
    ### Instead of simply truncating/repeating the reference profile based
    ### on length, match the reference according to:
    ###       month + day + hour + minute
    ### This allows a 2023 reference profile to be applied to another year.
    reference_shape = reference.copy()
    reference_shape.index = pd.MultiIndex.from_arrays(
        [
            reference_shape.index.month,
            reference_shape.index.day,
            reference_shape.index.hour,
            reference_shape.index.minute
        ],
        names=[
            "month",
            "day",
            "hour",
            "minute"
        ]
    )
    ### Remove duplicated timestamps/components if necessary
    reference_shape = reference_shape[
        ~reference_shape.index.duplicated(
            keep="first"
        )
    ]
    ### 5. Map reference profile onto target timestamps
    target_keys = pd.MultiIndex.from_arrays(
        [
            dhw.index.month,
            dhw.index.day,
            dhw.index.hour,
            dhw.index.minute
        ],
        names=[
            "month",
            "day",
            "hour",
            "minute"
        ]
    )
    reshaped_shape = reference_shape.reindex(
        target_keys
    )
    ### 6. Handle missing timestamps
    ### The main case is a leap-year target containing February 29,
    ### while the 2023 reference does not contain February 29.
    ### Missing values are filled using interpolation over the sequence
    ### of the reference profile.
    if reshaped_shape.isna().any():
        reference_values = reference_shape.to_numpy(
            dtype=float
        )
        valid_reference_values = reference_values[
            ~np.isnan(reference_values)
        ]
        if len(valid_reference_values) > 0:
            missing_positions = np.isnan(
                reshaped_shape.to_numpy(
                    dtype=float
                )
            )
            filled_values = reshaped_shape.to_numpy(
                dtype=float
            )
            filled_values[missing_positions] = np.mean(
                valid_reference_values
            )
            reshaped_shape = pd.Series(
                filled_values,
                index=target_keys
            )
    ### Restore target datetime index
    reshaped_shape.index = dhw.index
    ### Remove negative values
    reshaped_shape = reshaped_shape.clip(
        lower=0.0
    )
    ### 7. Normalize the reference shape
    reference_sum = reshaped_shape.sum()
    if reference_sum <= 0:
        print(
            "   ⚠️ Mapped reference profile has zero energy. "
            "Original DHW profile will be retained."
        )
        return dhw.copy()
    normalized_shape = (
        reshaped_shape / reference_sum
    )
    ### 8. Rescale reference shape to original DHW energy
    reshaped_dhw = (
        normalized_shape * target_energy
    )
    ### 9. Final energy correction
    reshaped_energy = reshaped_dhw.sum()
    if reshaped_energy > 0:
        reshaped_dhw *= (
            target_energy / reshaped_energy
        )
    ### Make sure index is exactly the original target index
    reshaped_dhw.index = dhw.index
    return reshaped_dhw
### MAIN PROCESSING:
### RESHAPE NODE-PREFIXED DHW DEMAND
for country_code, heat_dict in country_loads_time_series.items():
    ### 1. Load country-specific 2023 reference profile
    reference_profile_path = (
        f"{root_dir}/Boundary_Sector/"
        f"BoundarySectorDemand/{country_code}/2023.csv"
    )
    if not os.path.exists(reference_profile_path):
        print(
            f"⚠️ [{country_code}] Reference file not found:"
            f"\n   {reference_profile_path}"
        )
        continue
    ### Read reference profile
    ref_df = pd.read_csv(
        reference_profile_path,
        index_col=0,
        parse_dates=True
    )
    ### Use first data column as reference DHW profile
    ref_series = ref_df.iloc[:, 0].copy()
    ### Ensure datetime index
    if not isinstance(
        ref_series.index,
        pd.DatetimeIndex
    ):
        ref_series.index = pd.to_datetime(
            ref_series.index
        )
    ### Remove timezone if present
    if ref_series.index.tz is not None:
        ref_series.index = (
            ref_series.index.tz_localize(None)
        )
    print(
        f"\n📌 [{country_code}] "
        f"Reference DHW profile loaded:"
        f" {len(ref_series)} observations"
    )
    ### 2. Process every heat category
    for heat_category in all_heat_loads:
        if heat_category not in heat_dict:
            continue
        df = heat_dict[heat_category]
        if df.empty:
            continue
        ### 3. Find node-prefixed DHW columns created by the seasonal split
        ### Example:
        ### BE0 0 DHW_demand
        ### BE0 1 DHW_demand
        ### BE0 2 DHW_demand
        dhw_cols = [
            col
            for col in df.columns
            if str(col).endswith(
                "DHW_demand"
            )
        ]
        if not dhw_cols:
            continue
        ### 4. Extract node prefixes
        nodes_in_df = set()
        for col in dhw_cols:
            col_string = str(col)
            node_match = re.match(
                r"^([A-Z]{2}\d+\s+\d+)",
                col_string
            )
            if node_match:
                nodes_in_df.add(
                    node_match.group(1).strip()
                )
        ### Fallback
        if not nodes_in_df:
            nodes_in_df = {
                country_code
            }
        ### 5. Process every node independently
        processed_nodes = []
        for node in sorted(nodes_in_df):
            ### Select the DHW column belonging to this node
            node_cols = [
                col
                for col in dhw_cols
                if str(col).startswith(node)
            ]
            if not node_cols:
                continue
            ### There should normally be one DHW_demand column per node.
            ### If there is more than one, aggregate them.
            total_dhw = (
                df[node_cols]
                .apply(
                    pd.to_numeric,
                    errors="coerce"
                )
                .fillna(0.0)
                .sum(axis=1)
            )
            ### 6. Reshape DHW using country-specific reference profile
            if total_dhw.sum() > 0:
                reshaped_dhw = reshape_dhw_series(
                    total_dhw,
                    ref_series
                )
            else:
                reshaped_dhw = pd.Series(
                    0.0,
                    index=df.index
                )
            ### 7. Add reshaped DHW column
            reshaped_column_name = (
                f"{node} DHW_demand_reshaped"
            )
            df[
                reshaped_column_name
            ] = reshaped_dhw
            ### 8. Energy balance check
            original_energy = total_dhw.sum()
            reshaped_energy = (
                reshaped_dhw.sum()
            )
            balanced = np.isclose(
                original_energy,
                reshaped_energy,
                rtol=1e-9,
                atol=1e-9
            )
            print(
                f"   ✅ [{country_code}] "
                f"[{heat_category}] "
                f"[{node}] "
                f"DHW reshaped | "
                f"Original: {original_energy:.6f} | "
                f"Reshaped: {reshaped_energy:.6f} | "
                f"Balanced: {balanced}"
            )
            processed_nodes.append(node)
        ### 9. Report processed nodes
        if processed_nodes:
            print(
                f"✅ [{country_code}] "
                f"[{heat_category}] "
                f"Reshaped DHW for nodes: "
                f"{', '.join(processed_nodes)}"
            )
        else:
            print(
                f"⚠️ [{country_code}] "
                f"[{heat_category}] "
                f"No DHW nodes could be processed."
            )
print(
    "\n============================================================"
)
print(
    "DONE: Node-prefixed DHW demand profiles have been reshaped "
    "using the country-specific 2023 reference profiles."
)
print(
    "Original DHW columns have been preserved."
)
print(
    "New columns ending in 'DHW_demand_reshaped' have been added."
)
print(
    "============================================================"
)


📌 [BE] Reference DHW profile loaded: 8760 observations
   ✅ [BE] [urban central heat] [BE1 0] DHW reshaped | Original: 585322.081109 | Reshaped: 585322.081109 | Balanced: True
✅ [BE] [urban central heat] Reshaped DHW for nodes: BE1 0
   ✅ [BE] [services urban decentral heat] [BE1 0] DHW reshaped | Original: 1295350.073452 | Reshaped: 1295350.073452 | Balanced: True
✅ [BE] [services urban decentral heat] Reshaped DHW for nodes: BE1 0
   ✅ [BE] [services rural heat] [BE1 0] DHW reshaped | Original: 483540.257937 | Reshaped: 483540.257937 | Balanced: True
✅ [BE] [services rural heat] Reshaped DHW for nodes: BE1 0
   ✅ [BE] [residential rural heat] [BE1 0] DHW reshaped | Original: 184375.794817 | Reshaped: 184375.794817 | Balanced: True
✅ [BE] [residential rural heat] Reshaped DHW for nodes: BE1 0
   ✅ [BE] [residential urban decentral heat] [BE1 0] DHW reshaped | Original: 3629042.637170 | Reshaped: 3629042.637170 | Balanced: True
✅ [BE] [residential urban decentral heat] Reshaped DHW fo

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        F-3. Converting Thermal Demand to Electrical Demand Using Resistive Heater Efficiencies
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process converts the <b>thermal demand</b> (Space Heating and DHW) into <b>electrical demand</b> by applying the <b>average efficiency</b> of resistive heaters for each node. <br>The conversion uses static efficiency values from the country-specific PyPSA CSV files to calculate the electrical input required to meet the thermal demand.<br>
    <b>Conversion Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each country, heat load category, and demand type (SpaceHeating_demand, DHW_demand_reshaped), the script identifies all <b>resistive heater proportion columns</b> belonging to the same node.<br>
            For each resistive heater asset, it finds the matching bus and retrieves the corresponding <b>static efficiency</b> from the metadata.
        <br>The <b>average efficiency</b> of all resistive heaters in that node is calculated, and the thermal demand is divided by this average to obtain the <b>electrical demand</b>:
            <div style=" text-align: center; font-size: 14px;">
        <b>Electrical Demand = Thermal Demand / Average Efficiency</b><br>
            </div>
        The resulting electrical demand profiles are added as new columns (<b>{node} {demand_type}_/Efficiency</b>), ensuring that resistive heater loads are correctly represented as electrical consumption in Dispa-SET.
        </div>
    </div>
</div>

In [33]:
### MAP PYPSA BUS COLUMNS TO THEIR CORRESPONDING EFFICIENCY PARAMETERS
# =============================================================================
bus_to_eff_map = {
    "bus":  "efficiency",
    "bus0": "efficiency",
    "bus1": "efficiency",
    "bus2": "efficiency2",
    "bus3": "efficiency3",
    "bus4": "efficiency4",
}
# =============================================================================
### DEMAND TYPES TO PROCESS
# =============================================================================
demand_types = [
    "SpaceHeating_demand",
    "DHW_demand_reshaped",
]
# =============================================================================
### IN-PLACE EFFICIENCY CONVERSION
### ONLY RESISTIVE HEATERS ARE CONSIDERED
### The required power is calculated as:
###     Electrical/Input demand = Thermal demand / efficiency
### Efficiencies are obtained from the country-specific PyPSA CSV files.
### No country_dataframes
### No n.links_t
### No COP
for country, loads_dict in country_loads_time_series.items():
    print(f"\n--- Processing Country: {country} ---")
    ### LOAD COUNTRY-SPECIFIC POWER-PLANT METADATA
    power_plants_file = os.path.join(
        pypsa_raw_data_folder_path,
        pypsa_scenario,
        "PowerPlants",
        data_target_year,
        country,
        f"pypsa_power_plants_{country}_{data_target_year}.csv"
    )
    print(f"  Loading metadata from:")
    print(f"  {power_plants_file}")
    master_df = pd.read_csv(power_plants_file)
    master_df = master_df.set_index("name")
    ### PROCESS EACH HEAT CATEGORY
    for current_heat_load, ts_df in loads_dict.items():
        if ts_df.empty:
            continue
        ### PROCESS SPACE HEATING AND DHW
        for demand_type in demand_types:
            ### FIND DEMAND COLUMNS
            ### Examples:
            ### BE1 0 SpaceHeating_demand
            ### BE1 1 SpaceHeating_demand
            ### BE1 0 DHW_demand_reshaped
            ### BE1 1 DHW_demand_reshaped
            demand_cols = [
                col
                for col in ts_df.columns
                if col.endswith(demand_type)
            ]
            if not demand_cols:
                continue
            ### PROCESS EACH NODE
            for demand_col in demand_cols:
                ### EXTRACT NODE PREFIX
                ### Example:
                ### "BE1 0 SpaceHeating_demand"
                ### becomes:
                ### "BE1 0"
                node_prefix = demand_col.replace(
                    f" {demand_type}",
                    ""
                )
                ### FIND RESISTIVE-HEATER PROPORTION COLUMNS
                ### BELONGING TO THIS NODE
                node_matching_cols = [
                    col
                    for col in ts_df.columns
                    if (
                        "resistive heater" in col.lower()
                        and "_proportion" in col.lower()
                        and col.startswith(node_prefix)
                    )
                ]
                if not node_matching_cols:
                    print(
                        f"  ⚠️ [{current_heat_load}] "
                        f"{demand_type} | "
                        f"Node: {node_prefix} | "
                        f"No resistive-heater proportion columns found."
                    )
                    continue
                ### COLLECT RESISTIVE-HEATER EFFICIENCIES
                efficiency_objects = []
                for proportion_col in node_matching_cols:
                    ### REMOVE "_proportion" FROM COLUMN NAME
                    ### Example:
                    ### BE1 0 residential rural resistive heater-2030_proportion
                    ### becomes:
                    ### BE1 0 residential rural resistive heater-2030
                    asset_name = proportion_col[:-11]
                    ### FIND ASSET IN COUNTRY CSV
                    if asset_name not in master_df.index:
                        print(
                            f"  ⚠️ [{current_heat_load}] "
                            f"[{node_prefix}] "
                            f"Asset '{asset_name}' "
                            f"not found in metadata."
                        )
                        continue
                    asset_row = master_df.loc[asset_name]
                    ### FIND THE BUS ASSOCIATED WITH THE CURRENT HEAT LOAD
                    ### This determines which efficiency column must be used:
                    ### bus  -> efficiency
                    ### bus0 -> efficiency
                    ### bus1 -> efficiency
                    ### bus2 -> efficiency2
                    ### bus3 -> efficiency3
                    ### bus4 -> efficiency4
                    matched_eff_col = None
                    for bus_col, eff_col in bus_to_eff_map.items():
                        if bus_col not in asset_row:
                            continue
                        bus_value = asset_row[bus_col]
                        if pd.isna(bus_value):
                            continue
                        if current_heat_load in str(bus_value):
                            matched_eff_col = eff_col
                            break
                    if matched_eff_col is None:
                        print(
                            f"  ⚠️ [{current_heat_load}] "
                            f"[{node_prefix}] "
                            f"No matching heat bus found for "
                            f"'{asset_name}'."
                        )
                        continue
                    ### GET STATIC EFFICIENCY
                    if matched_eff_col not in asset_row:
                        continue
                    efficiency_value = asset_row[matched_eff_col]
                    if pd.isna(efficiency_value):
                        continue
                    efficiency_value = float(efficiency_value)
                    if efficiency_value <= 0:
                        print(
                            f"  ⚠️ [{current_heat_load}] "
                            f"[{node_prefix}] "
                            f"Invalid efficiency "
                            f"{efficiency_value} for '{asset_name}'."
                        )
                        continue
                    ### STORE EFFICIENCY
                    efficiency_objects.append(
                        efficiency_value
                    )
                ### CALCULATE AVERAGE RESISTIVE-HEATER EFFICIENCY
                out_col = (
                    f"{node_prefix} "
                    f"{demand_type}_/Efficiency"
                )
                if efficiency_objects:
                    ### SIMPLE ARITHMETIC MEAN OF THE RESISTIVE-HEATER
                    ### EFFICIENCIES
                    average_efficiency = (
                        sum(efficiency_objects)
                        / len(efficiency_objects)
                    )
                    ### CONVERT THERMAL DEMAND INTO ELECTRICAL DEMAND
                    ts_df[out_col] = (
                        ts_df[demand_col]
                        / average_efficiency
                    )
                    print(
                        f"  ✅ [{current_heat_load}] "
                        f"{demand_type} | "
                        f"Node: {node_prefix} | "
                        f"Resistive heaters: "
                        f"{len(efficiency_objects)} | "
                        f"Average efficiency: "
                        f"{average_efficiency:.3f}"
                    )
                else:
                    print(
                        f"  ⚠️ [{current_heat_load}] "
                        f"{demand_type} | "
                        f"Node: {node_prefix} | "
                        f"No valid resistive-heater efficiencies found."
                    )
### FINISHED
print(
    "\nDone! Static efficiency conversion for "
    "Space Heating and DHW using resistive heaters only."
)


--- Processing Country: BE ---
  Loading metadata from:
  /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/PowerPlants/2030/BE/pypsa_power_plants_BE_2030.csv
  ✅ [urban central heat] SpaceHeating_demand | Node: BE1 0 | Resistive heaters: 1 | Average efficiency: 0.990
  ✅ [urban central heat] DHW_demand_reshaped | Node: BE1 0 | Resistive heaters: 1 | Average efficiency: 0.990
  ✅ [services urban decentral heat] SpaceHeating_demand | Node: BE1 0 | Resistive heaters: 3 | Average efficiency: 0.900
  ✅ [services urban decentral heat] DHW_demand_reshaped | Node: BE1 0 | Resistive heaters: 3 | Average efficiency: 0.900
  ✅ [services rural heat] SpaceHeating_demand | Node: BE1 0 | Resistive heaters: 3 | Average efficiency: 0.900
  ✅ [services rural heat] DHW_demand_reshaped | Node: BE1 0 | Resistive heaters: 3 | Average efficiency: 0.900
  ✅ [residential rural heat] SpaceHeating_demand | Node: BE1 0 | Resistive heaters: 3 | Average efficiency: 0.900
  ✅ [residential rural heat] DH

<div style= "background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099; " >
 <span style= "font-weight: bold; font-size: 16px; " >
Section G Overview: Space Heating Aggregation & VPP Sector Load
 </span >
 <div style= "border-top: 1px solid #000099; padding: 10px; " >
This section finalizes the VPP power load calculations.<br> It extracts and sums the electrical equivalency of space heating across all nodes. <br>If VPP mode is enabled, it adds this space heating equivalency to the baseline Sector Power Load to create the comprehensive VPP Sector Power Load dictionary.
 </div >
 </div >
 <div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        G-1. Extracting and Summing Space Heating Consumption Electric Equivalency
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process extracts all <b>SpaceHeating_demand_/Efficiency</b> columns from the <b>country_loads_time_series</b> dictionary, sums them horizontally for each country, and stores the result in a new dictionary (<b>space_heating_consumption_elect_equivalency</b>).<br> These columns represent the <b>electric equivalency of space heating consumption</b> for each node.<br>
    <b>Extraction and Aggregation Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each country, the script iterates through all heat load categories and identifies columns containing <b>"SpaceHeating_demand_/Efficiency"</b>.<br> These columns represent the electrical input required for space heating (accounting for efficiency).<br> The found columns are collected from all heat categories and then <b>concatenated horizontally</b>.<br> The total space heating consumption is calculated by <b>summing all columns along the row axis</b> (<b>df.sum(axis=1)</b>).<br>
        The resulting dictionary contains one DataFrame per country with a single column named <b>"space_heating_consumption_elect_equivalency"</b>.
        </div>
    </div>
</div>

In [34]:
space_heating_consumption_elect_equivalency = {}
for country, heat_loads_dict in country_loads_time_series.items():
    country_space_heating_dfs = []
    for heat_category, df in heat_loads_dict.items():
        # FIND SpaceHeating_demand_/Efficiency COLUMNS
        efficiency_cols = [
            col
            for col in df.columns
            if "SpaceHeating_demand_/Efficiency" in str(col)
        ]
        # EXTRACT FOUND COLUMNS
        if efficiency_cols:
            country_space_heating_dfs.append(
                df[efficiency_cols]
            )
    # AGGREGATE ALL FOUND COLUMNS
    if country_space_heating_dfs:
        combined_space_heating = pd.concat(
            country_space_heating_dfs,
            axis=1
        )
        total_space_heating = combined_space_heating.sum(axis=1)
        space_heating_consumption_elect_equivalency[country] = (
            total_space_heating.to_frame(
                name="space_heating_consumption_elect_equivalency"
            )
        )
    else:
        # Keep the country's time index even if no columns were found
        first_df = next(iter(heat_loads_dict.values()))
        space_heating_consumption_elect_equivalency[country] = (
            pd.DataFrame(
                index=first_df.index,
                columns=[
                    "space_heating_consumption_elect_equivalency"
                ]
            )
        )
        space_heating_consumption_elect_equivalency[country] = (
            space_heating_consumption_elect_equivalency[country].fillna(0.0)
        )
    # Make sure the index is called snapshot
    space_heating_consumption_elect_equivalency[country].index.name = (
        "snapshot"
    )
# INSPECTION
print("\nProcess completed.")
for country, df in space_heating_consumption_elect_equivalency.items():
    print(
        f" {country}: "
        f"{df.shape[0]} snapshots, "
        f"{df.shape[1]} column"
    )
    print(
        df.head(2)
    )


Process completed.
 BE: 8760 snapshots, 1 column
                     space_heating_consumption_elect_equivalency
snapshot                                                        
2013-01-01 00:00:00                                  3445.828820
2013-01-01 01:00:00                                  3497.694145
 FR: 8760 snapshots, 1 column
                     space_heating_consumption_elect_equivalency
snapshot                                                        
2013-01-01 00:00:00                                 20252.435589
2013-01-01 01:00:00                                 20748.994932
 DE: 8760 snapshots, 1 column
                     space_heating_consumption_elect_equivalency
snapshot                                                        
2013-01-01 00:00:00                                  8342.643823
2013-01-01 01:00:00                                  8287.691133
 NL: 8760 snapshots, 1 column
                     space_heating_consumption_elect_equivalency
snapshot       

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        G-2. Creating VPP Sector Power Load
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process creates the <b>VPP Sector Power Load</b> dictionary by starting with the original <b>Sector_Power_Load</b> and <b>conditionally adding space heating consumption</b> when VPP mode is enabled. The result is stored in a new dictionary (<b>vpp_Sector_Power_Load</b>).<br>
    <b>Creation Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each country, the script copies the original <b>Sector_Power_Load</b> DataFrame. <br>
        If <b>VPP = True</b>, it retrieves the space heating consumption DataFrame and validates that both DataFrames have the <b>same number of snapshots</b> and <b>identical snapshot indexes</b>.<br>
        The space heating column is then added to the Sector_Power_Load column. If <b>VPP = False</b>, no addition is performed.<br>
        The resulting dictionary contains one DataFrame per country with the updated <b>Sector_Power_Load</b> column.
        </div>
    </div>
</div>

In [35]:
# CREATE VPP SECTOR POWER LOAD
vpp_Sector_Power_Load = {}
for country in Sector_Power_Load:
    # COPY ORIGINAL SECTOR POWER LOAD
    sector_df = Sector_Power_Load[country].copy()
    # IF VPP IS ACTIVE, ADD SPACE HEATING
    if VPP:
        space_heating_df = (
            space_heating_consumption_elect_equivalency[country]
            .copy()
        )
        # NORMALIZE SNAPSHOT INDEXES
        sector_index = pd.to_datetime(sector_df.index)
        space_heating_index = pd.to_datetime(
            space_heating_df.index
        )
        # Remove timezone if present
        if sector_index.tz is not None:
            sector_index = sector_index.tz_localize(None)
        if space_heating_index.tz is not None:
            space_heating_index = space_heating_index.tz_localize(None)
        sector_df.index = sector_index
        space_heating_df.index = space_heating_index
        sector_df.index.name = "snapshot"
        space_heating_df.index.name = "snapshot"
        # CHECK SAME NUMBER OF SNAPSHOTS
        if len(sector_df) != len(space_heating_df):
            raise ValueError(
                f"{country}: different number of snapshots. "
                f"Sector_Power_Load = {len(sector_df)}, "
                f"Space Heating = {len(space_heating_df)}"
            )
        # CHECK SAME SNAPSHOT INDEX
        if not sector_df.index.equals(space_heating_df.index):
            raise ValueError(
                f"{country}: snapshot indexes are not identical."
            )
        # SUM INTO THE SAME COLUMN
        sector_df["Sector_Power_Load"] = (
            sector_df["Sector_Power_Load"]
            + space_heating_df[
                "space_heating_consumption_elect_equivalency"
            ]
        )
    # SAVE RESULT INTO NEW DICTIONARY
    vpp_Sector_Power_Load[country] = sector_df
    # INSPECTION
    print(
        f"\n{country}: "
        f"{len(vpp_Sector_Power_Load[country])} snapshots"
    )
    print(
        vpp_Sector_Power_Load[country].head(2)
    )
# FINAL MESSAGE
if VPP:
    print(
        "\nVPP activated: "
        "space heating was added to vpp_Sector_Power_Load."
    )
else:
    print(
        "\nVPP deactivated: "
        "vpp_Sector_Power_Load is equal to Sector_Power_Load."
    )


BE: 8760 snapshots
                     Sector_Power_Load
snapshot                              
2013-01-01 00:00:00       22961.222891
2013-01-01 01:00:00       23364.638164

FR: 8760 snapshots
                     Sector_Power_Load
snapshot                              
2013-01-01 00:00:00      104519.036188
2013-01-01 01:00:00      104019.803187

DE: 8760 snapshots
                     Sector_Power_Load
snapshot                              
2013-01-01 00:00:00       98591.415053
2013-01-01 01:00:00       97134.133069

NL: 8760 snapshots
                     Sector_Power_Load
snapshot                              
2013-01-01 00:00:00       25936.857853
2013-01-01 01:00:00       25355.925102

UK: 8760 snapshots
                     Sector_Power_Load
snapshot                              
2013-01-01 00:00:00       68667.940928
2013-01-01 01:00:00       63767.863324

VPP deactivated: vpp_Sector_Power_Load is equal to Sector_Power_Load.


<div style= "background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099; " >
 <span style= "font-weight: bold; font-size: 16px; " >
Section H Overview: Target Year Adjustment & Power Load Export
 </span >
 <div style= "border-top: 1px solid #000099; padding: 10px; " >
This section prepares the power load time series for export.<br> It adjusts the DataFrames to match the exact number of rows required for the target year, intelligently handling leap year differences (February 29). <br>It then saves the files with dynamic VPP suffixes and copies them to the appropriate standard or VPP directories based on the zone configuration.
 </div >
 </div >
 <div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        H-1. Adjusting Selected Load Dictionary to Target Year with Leap Year Handling
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process adjusts the <b>selected load dictionary</b> (either <b>Sector_Power_Load</b> or <b>vpp_Sector_Power_Load</b>) to have the correct number of rows for the target year, handling <b>leap year</b> differences (February 29).<br>It ensures that all DataFrames have the exact number of timesteps expected for the target year.<br>
    <b>Adjustment Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        The script first determines if the target year is a leap year and calculates the expected number of rows (365 or 366 days × steps per day). It then selects the appropriate load dictionary based on <b>VPP</b> mode:
            <div style="margin-left: 2em;">
        <b>VPP = True</b> → Processes <b>vpp_Sector_Power_Load</b><br>
        <b>VPP = False</b> → Processes <b>Sector_Power_Load</b>
            </div>
        For each zone, it adjusts the DataFrame to match the target year:
            <div style="margin-left: 2em;">
        <b>If target year has more rows</b> (leap year) → Duplicate February 28 data to create a February 29 row.<br>
        <b>If target year has fewer rows</b> (non-leap year) → Remove February 29 data.
            </div>
        The process uses a <b>temporary leap year 2000 index</b> to safely handle February 29 during processing, then reindexes to the target year with a UTC timezone.<br>
        After adjustment, all DataFrames have the correct number of rows for the target year and are properly indexed with a UTC datetime index named "snapshot".
        </div>
    </div>
</div>

In [36]:
# TARGET YEAR SETTINGS
target_year = int(data_target_year)
is_leap_year = calendar.isleap(target_year)
# =============================================================================
steps_per_day = {
    "1h": 24,
    "30min": 48,
    "15min": 96
}
# =============================================================================
expected_rows = (
    366 if is_leap_year else 365
) * steps_per_day[data_target_time_step]

print(f"Target year   : {target_year}")
print(f"Leap year     : {is_leap_year}")
print(f"Expected rows : {expected_rows}")
# SELECT LOAD DICTIONARY DEPENDING ON VPP
if VPP:
    load_dictionary = vpp_Sector_Power_Load
    print("\nVPP enabled -> processing vpp_Sector_Power_Load")
else:
    load_dictionary = Sector_Power_Load
    print("\nVPP disabled -> processing Sector_Power_Load")
# ADJUST ALL ZONES
for zone, df in load_dictionary.items():
    df = df.copy()
    actual_rows = len(df)
    print(f"\nProcessing {zone}")
    print(f"Actual rows   : {actual_rows}")
    print(f"Expected rows : {expected_rows}")
    # CREATE TEMPORARY INDEX USING LEAP YEAR 2000
    # 2000 is a leap year, so Feb-29 can exist if needed.
    base_freq = {
        "1h": "h",
        "30min": "30min",
        "15min": "15min"
    }[data_target_time_step]
    temp_index = pd.date_range(
        start="2000-01-01 00:00:00",
        periods=actual_rows,
        freq=base_freq
    )
    df.index = temp_index
    # TARGET YEAR HAS MORE ROWS
    # ADD FEBRUARY 29
    if expected_rows > actual_rows:
        feb28_mask = (
            (temp_index.month == 2)
            & (temp_index.day == 28)
        )
        feb28 = df.loc[feb28_mask].copy()
        feb29_index = feb28.index + pd.Timedelta(days=1)
        feb29 = feb28.copy()
        feb29.index = feb29_index
        df = pd.concat(
            [df, feb29]
        )
        df = df.sort_index()
    # TARGET YEAR HAS FEWER ROWS
    # REMOVE FEBRUARY 29
    elif expected_rows < actual_rows:
        df = df[
            ~(
                (df.index.month == 2)
                & (df.index.day == 29)
            )
        ]
    # CREATE FINAL UTC INDEX
    final_index = pd.date_range(
        start=f"{target_year}-01-01 00:00:00",
        periods=len(df),
        freq=base_freq,
        tz="UTC"
    )
    df.index = final_index
    df.index.name = "snapshot"
    # SAVE BACK TO THE SELECTED DICTIONARY
    load_dictionary[zone] = df
    print(f"Final rows    : {len(df)}")
# FINAL MESSAGE
if VPP:
    print(
        "\nVPP enabled -> "
        "vpp_Sector_Power_Load was adjusted to the target year."
    )
else:
    print(
        "\nVPP disabled -> "
        "Sector_Power_Load was adjusted to the target year."
    )

Target year   : 2030
Leap year     : False
Expected rows : 8760

VPP disabled -> processing Sector_Power_Load

Processing BE
Actual rows   : 8760
Expected rows : 8760
Final rows    : 8760

Processing FR
Actual rows   : 8760
Expected rows : 8760
Final rows    : 8760

Processing DE
Actual rows   : 8760
Expected rows : 8760
Final rows    : 8760

Processing NL
Actual rows   : 8760
Expected rows : 8760
Final rows    : 8760

Processing UK
Actual rows   : 8760
Expected rows : 8760
Final rows    : 8760

VPP disabled -> Sector_Power_Load was adjusted to the target year.


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        H-2. Saving Load Data with Dynamic File Suffix Based on VPP Mode
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process exports the <b>selected load dictionary</b> (either <b>Sector_Power_Load</b> or <b>vpp_Sector_Power_Load</b>) to CSV files with <b>automatic time-step detection</b> and a <b>dynamic file suffix</b> based on VPP mode.<br> When VPP is enabled, files are saved with a <b>_vpp</b> suffix to distinguish them from standard load files.<br>
    <b>Export Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        The script first selects the appropriate load dictionary and file suffix based on <b>VPP</b> mode:
            <div style="margin-left: 2em;">
        <b>VPP = True</b> → Uses <b>vpp_Sector_Power_Load</b> with <b>_vpp</b> suffix<br>
        <b>VPP = False</b> → Uses <b>Sector_Power_Load</b> with <b>no suffix</b>
            </div>
        For each zone, it calculates the time difference between the first two timestamps to determine the time-step subfolder (1h, 30min, or 15min). The DataFrame is then saved as <b>{data_target_year}{file_suffix}.csv</b> in the appropriate subfolder, with <b>index=True</b> and <b>header=False</b>.<br>
        This ensures that load data is correctly organized by time resolution for Dispa-SET compatibility.
        </div>
    </div>
</div>

In [37]:
# SELECT LOAD DICTIONARY AND FILE SUFFIX
if VPP:
    load_dictionary = vpp_Sector_Power_Load
    file_suffix = "_vpp"
    print("\nVPP enabled -> using vpp_Sector_Power_Load")
else:
    load_dictionary = Sector_Power_Load
    file_suffix = ""
    print("\nVPP disabled -> using Sector_Power_Load")
# SAVE LOAD DATA FOR EACH ZONE
for zone in zone_names:
    # Get the DataFrame for the current zone
    df = load_dictionary[zone]
    # EXTRACT THE TIME STEP FROM THE INDEX
    time_index = df.index
    if len(time_index) >= 2:
        # Calculate the time difference between
        # the first two timestamps
        time_diff = (
            pd.to_datetime(time_index[1])
            - pd.to_datetime(time_index[0])
        )
        time_diff_minutes = time_diff.total_seconds() / 60
        # Determine the time-step subfolder
        if time_diff_minutes == 60:
            time_step_subfolder = "1h"
        elif time_diff_minutes == 30:
            time_step_subfolder = "30min"
        elif time_diff_minutes == 15:
            time_step_subfolder = "15min"
        else:
            raise ValueError(
                f"Unsupported time step: "
                f"{time_diff_minutes} minutes"
            )
    else:
        raise ValueError(
            f"Zone {zone}: insufficient data points "
            f"to determine time step."
        )
    # DEFINE SAVE PATH
    save_path = os.path.join(
        loads_pypsa_formated_data_folder_path,
        zone,
        time_step_subfolder
    )
    # Create directory if it does not exist
    os.makedirs(
        save_path,
        exist_ok=True
    )
    # DEFINE FILE NAME
    file_name = (
        f"{data_target_year}{file_suffix}.csv"
    )
    file_path = os.path.join(
        save_path,
        file_name
    )
    # SAVE DATAFRAME
    df.to_csv(
        file_path,
        index=True,
        header=False
    )
    # DISPLAY SAVED FILE INFORMATION
    print(f"Zone: {zone}")
    print(f"Time step: {time_step_subfolder}")
    print(f"Saved file: {file_path}")
    print("-" * 80)


VPP disabled -> using Sector_Power_Load
Zone: BE
Time step: 1h
Saved file: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/Load_RealTime/BE/1h/2030.csv
--------------------------------------------------------------------------------
Zone: FR
Time step: 1h
Saved file: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/Load_RealTime/FR/1h/2030.csv
--------------------------------------------------------------------------------
Zone: DE
Time step: 1h
Saved file: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/Load_RealTime/DE/1h/2030.csv
--------------------------------------------------------------------------------
Zone: NL
Time step: 1h
Saved file: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/Load_RealTime/NL/1h/2030.csv
--------------------------------------------------------------------------------
Zone: UK
Time step: 1h
Saved file: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/Load_RealTime/UK/1h/2030.csv
-----

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        H-3. Copying VPP Power Sector Load Files with VPP Zone Handling
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process copies <b>load CSV files</b> from the source directory to the destination directory with <b>detailed VPP zone handling</b>.<br>
        It distinguishes between <b>VPP zones</b> and <b>non-VPP zones</b>, and behaves differently depending on whether VPP mode is enabled.<br>
    <b>Copy Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        The script first determines which zones are in <b>non_vpp_zones</b> (zones in zone_names but NOT in vpp_zone_names).<br>
        It then handles two scenarios:<br>
            <div style="margin-left: 2em;">
        <b>When VPP = True:</b><br>
                <div style="margin-left: 2em;">
        <li>VPP zones → Copy <code>{year}_vpp.csv</code> to <code>{year}.csv</code></li>
        <li>Non-VPP zones → Copy <code>{year}.csv</code> to <code>{year}.csv</code></li>
                </div>
        <b>When VPP = False:</b>
                <div style="margin-left: 2em;">
        <li>Only non-VPP zones are processed (VPP zones are skipped)<br></li>
        <li>Copy <code>{year}.csv</code> to <code>{year}.csv</code></li>
                </div>
            </div>
This ensures that all load files are correctly organized for Dispa-SET compatibility.
    </div>
</div>

In [38]:
# SOURCE AND DESTINATION ROOTS
source_root = Path(
    loads_pypsa_formated_data_folder_path
)
destination_root = Path(
    vpp_loads_pypsa_formated_data_folder_path
)
# DETERMINE ZONE GROUPS
# Zones that are in zone_names but NOT in vpp_zone_names
non_vpp_zones = [
    zone for zone in zone_names
    if zone not in vpp_zone_names
]
# INITIALIZE COPY SUMMARY
copied_files = []
# VPP = TRUE
if VPP:
    print("\nVPP enabled")
    print("-" * 80)
    # 1. VPP ZONES
    # Source:
    #     2030_vpp.csv
    # Destination:
    #     2030.csv
    print("\nProcessing VPP zones...")
    for zone in vpp_zone_names:
        # Make sure we only process zones that are
        # actually part of the selected zone_names
        if zone not in zone_names:
            continue
        # TIME STEP
        source_df = vpp_Sector_Power_Load[zone]
        time_index = source_df.index
        if len(time_index) >= 2:
            time_diff = (
                pd.to_datetime(time_index[1])
                - pd.to_datetime(time_index[0])
            )
            time_diff_minutes = (
                time_diff.total_seconds() / 60
            )
            if time_diff_minutes == 60:
                time_step_subfolder = "1h"
            elif time_diff_minutes == 30:
                time_step_subfolder = "30min"
            elif time_diff_minutes == 15:
                time_step_subfolder = "15min"
            else:
                raise ValueError(
                    f"Zone {zone}: unsupported time step "
                    f"{time_diff_minutes} minutes."
                )
        else:
            raise ValueError(
                f"Zone {zone}: insufficient data points "
                f"to determine time step."
            )
        # SOURCE
        source_file = (
            source_root
            / zone
            / time_step_subfolder
            / f"{data_target_year}_vpp.csv"
        )
        # DESTINATION
        destination_file = (
            destination_root
            / zone
            / time_step_subfolder
            / f"{data_target_year}.csv"
        )
        # CREATE DESTINATION DIRECTORY
        destination_file.parent.mkdir(
            parents=True,
            exist_ok=True
        )
        # COPY / REPLACE
        shutil.copy2(
            source_file,
            destination_file
        )
        # RECORD
        copied_files.append(
            f"{zone}: "
            f"{source_file.name} -> "
            f"{destination_file.name} "
            f"(VPP, {time_step_subfolder})"
        )
    # 2. NON-VPP ZONES
    # Source:
    #     2030.csv
    # Destination:
    #     2030.csv
    print("\nProcessing non-VPP zones...")
    for zone in non_vpp_zones:
        # TIME STEP
        source_df = Sector_Power_Load[zone]
        time_index = source_df.index
        if len(time_index) >= 2:
            time_diff = (
                pd.to_datetime(time_index[1])
                - pd.to_datetime(time_index[0])
            )
            time_diff_minutes = (
                time_diff.total_seconds() / 60
            )
            if time_diff_minutes == 60:
                time_step_subfolder = "1h"
            elif time_diff_minutes == 30:
                time_step_subfolder = "30min"
            elif time_diff_minutes == 15:
                time_step_subfolder = "15min"
            else:
                raise ValueError(
                    f"Zone {zone}: unsupported time step "
                    f"{time_diff_minutes} minutes."
                )
        else:
            raise ValueError(
                f"Zone {zone}: insufficient data points "
                f"to determine time step."
            )
        # SOURCE
        source_file = (
            source_root
            / zone
            / time_step_subfolder
            / f"{data_target_year}.csv"
        )
        # DESTINATION
        destination_file = (
            destination_root
            / zone
            / time_step_subfolder
            / f"{data_target_year}.csv"
        )
        # CREATE DESTINATION DIRECTORY
        destination_file.parent.mkdir(
            parents=True,
            exist_ok=True
        )
        # COPY / REPLACE
        shutil.copy2(
            source_file,
            destination_file
        )
        # RECORD
        copied_files.append(
            f"{zone}: "
            f"{source_file.name} -> "
            f"{destination_file.name} "
            f"(Standard, {time_step_subfolder})"
        )
# VPP = FALSE
else:
    print("\nVPP disabled")
    print("-" * 80)
    # ONLY PROCESS NON-VPP ZONES
    # Example:
    # zone_names     = ["BE", "DE"]
    # vpp_zone_names = ["BE"]
    # non_vpp_zones  = ["DE"]
    # Therefore BE is NOT copied when VPP is False.
    print("\nProcessing non-VPP zones only...")
    for zone in non_vpp_zones:
        # TIME STEP
        source_df = Sector_Power_Load[zone]
        time_index = source_df.index
        if len(time_index) >= 2:
            time_diff = (
                pd.to_datetime(time_index[1])
                - pd.to_datetime(time_index[0])
            )
            time_diff_minutes = (
                time_diff.total_seconds() / 60
            )
            if time_diff_minutes == 60:
                time_step_subfolder = "1h"
            elif time_diff_minutes == 30:
                time_step_subfolder = "30min"
            elif time_diff_minutes == 15:
                time_step_subfolder = "15min"
            else:
                raise ValueError(
                    f"Zone {zone}: unsupported time step "
                    f"{time_diff_minutes} minutes."
                )
        else:
            raise ValueError(
                f"Zone {zone}: insufficient data points "
                f"to determine time step."
            )
        # SOURCE
        source_file = (
            source_root
            / zone
            / time_step_subfolder
            / f"{data_target_year}.csv"
        )
        # DESTINATION
        destination_file = (
            destination_root
            / zone
            / time_step_subfolder
            / f"{data_target_year}.csv"
        )
        # CREATE DESTINATION DIRECTORY
        destination_file.parent.mkdir(
            parents=True,
            exist_ok=True
        )
        # COPY / REPLACE
        shutil.copy2(
            source_file,
            destination_file
        )
        # RECORD
        copied_files.append(
            f"{zone}: "
            f"{source_file.name} -> "
            f"{destination_file.name} "
            f"(Standard, {time_step_subfolder})"
        )
# SUMMARY
print("\nCopy Summary")
print("-" * 80)
for item in copied_files:
    print(item)
print(
    f"\nTotal files copied: "
    f"{len(copied_files)}"
)


VPP disabled
--------------------------------------------------------------------------------

Processing non-VPP zones only...

Copy Summary
--------------------------------------------------------------------------------
FR: 2030.csv -> 2030.csv (Standard, 1h)
DE: 2030.csv -> 2030.csv (Standard, 1h)
NL: 2030.csv -> 2030.csv (Standard, 1h)
UK: 2030.csv -> 2030.csv (Standard, 1h)

Total files copied: 4


<div style= "background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099; " >
 <span style= "font-weight: bold; font-size: 16px; " >
Section I Overview: DHW Load Desegregation & Aggregation
 </span >
 <div style= "border-top: 1px solid #000099; padding: 10px; " >
This section organizes the Domestic Hot Water (DHW) demands for the VPP optimization. <br>It extracts all reshaped DHW columns from the heat load dictionaries, desegregates them by heat category for traceability, and then sums them horizontally to create a single, unified total DHW load time series per country.
 </div >
 </div >
 <div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        I-1. Creating DHW Load Desegregated Dictionary
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process extracts all <b>DHW_demand_reshaped</b> columns from the <b>country_loads_time_series</b> dictionary and organizes them into a <b>desegregated</b> structure.<br>
        Each heat load category's DHW columns are renamed with a suffix to preserve their origin, and all columns are combined into a single DataFrame per zone.<br>
    <b>Desegregation Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each zone, the script iterates through all heat load categories (subkeys) and identifies columns containing <b>"DHW_demand_reshaped"</b>.<br>For each category, it extracts the matching columns and renames them by appending the subkey (with spaces replaced by underscores) to the original column name.<br>All extracted DataFrames are then concatenated <b>horizontally</b> into a single DataFrame per zone.
         This desegregated structure provides a clear view of all DHW demand profiles organized by heat load category.
        </div>
    </div>
</div>

In [39]:
DHW_load_desegregated = {}
for zone in zone_names:
    zone_dhw_data = {}
    for subkey, df in country_loads_time_series[zone].items():
        # Find columns containing "DHW_demand_reshaped"
        dhw_columns = [
            column
            for column in df.columns
            if "DHW_demand_reshaped" in str(column)
        ]
        if len(dhw_columns) == 0:
            continue
        # Copy only the selected columns
        dhw_df = df[dhw_columns].copy()
        # Rename each selected column by adding the subkey
        dhw_df.columns = [
            f"{column}_{subkey.replace(' ', '_')}"
            for column in dhw_df.columns
        ]
        zone_dhw_data[subkey] = dhw_df
    # Combine all subkeys horizontally, keeping the vertical index
    if zone_dhw_data:
        DHW_load_desegregated[zone] = pd.concat(
            zone_dhw_data.values(),
            axis=1
        )
    else:
        DHW_load_desegregated[zone] = pd.DataFrame(
            index=country_loads_time_series[zone].index
        )
print("\nDHW_load_desegregated successfully created.\n")
for zone, df in DHW_load_desegregated.items():
    print(
        f"Zone: {zone} \n "
        f"Rows: {len(df):,} \n "
        f"Columns ({len(df.columns)}): {list(df.columns)}"
    )


DHW_load_desegregated successfully created.

Zone: BE 
 Rows: 8,760 
 Columns (10): ['BE1 0 DHW_demand_reshaped_urban_central_heat', 'BE1 0 DHW_demand_reshaped_/Efficiency_urban_central_heat', 'BE1 0 DHW_demand_reshaped_services_urban_decentral_heat', 'BE1 0 DHW_demand_reshaped_/Efficiency_services_urban_decentral_heat', 'BE1 0 DHW_demand_reshaped_services_rural_heat', 'BE1 0 DHW_demand_reshaped_/Efficiency_services_rural_heat', 'BE1 0 DHW_demand_reshaped_residential_rural_heat', 'BE1 0 DHW_demand_reshaped_/Efficiency_residential_rural_heat', 'BE1 0 DHW_demand_reshaped_residential_urban_decentral_heat', 'BE1 0 DHW_demand_reshaped_/Efficiency_residential_urban_decentral_heat']
Zone: FR 
 Rows: 8,760 
 Columns (10): ['FR1 0 DHW_demand_reshaped_urban_central_heat', 'FR1 0 DHW_demand_reshaped_/Efficiency_urban_central_heat', 'FR1 0 DHW_demand_reshaped_services_urban_decentral_heat', 'FR1 0 DHW_demand_reshaped_/Efficiency_services_urban_decentral_heat', 'FR1 0 DHW_demand_reshaped_services_

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        I-2. Creating DHW_load Dictionary (Total DHW Load)
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process creates the <b>DHW_load</b> dictionary by summing all columns of the <b>DHW_load_desegregated</b> DataFrame for each country.<br>
        The result is a <b>single time series</b> per country representing the <b>total Domestic Hot Water (DHW) load</b> at each snapshot.<br>
    <b>Aggregation Logic:</b>
        <div style="margin-left: 2em; font-size: 12px;">
        For each country, the script ensures that the <b>snapshot</b> is set as the DatetimeIndex.<br>
        It then selects all columns except "snapshot" and calculates the <b>row-wise sum</b> using <b>df.sum(axis=1)</b>.<br>
        The resulting total DHW load is stored as a single-column DataFrame named <b>"DHW_load"</b> with the snapshot as the index.<br>
        A summary report displays the number of DHW columns summed and the number of snapshots per country. 
        </div>
    </div>
</div>

In [40]:
# CREATE DHW_LOAD DICTIONARY
# For each country in zone_names:
#     DHW_load[country]
# is the sum of ALL columns in:
#     DHW_load_desegregated[country]
# The snapshot index is preserved.
DHW_load = {}
# PROCESS EACH COUNTRY
for country in zone_names:
    print(f"\n--- Processing DHW Load: {country} ---")
    # GET COUNTRY DATAFRAME
    dhw_desegregated_df = DHW_load_desegregated[country]
    # MAKE SURE SNAPSHOT IS THE INDEX
    if "snapshot" in dhw_desegregated_df.columns:
        dhw_desegregated_df = dhw_desegregated_df.copy()
        dhw_desegregated_df["snapshot"] = pd.to_datetime(
            dhw_desegregated_df["snapshot"]
        )
        dhw_desegregated_df = dhw_desegregated_df.set_index(
            "snapshot"
        )
    else:
        dhw_desegregated_df = dhw_desegregated_df.copy()
        dhw_desegregated_df.index = pd.to_datetime(
            dhw_desegregated_df.index
        )
    # SELECT ALL DATA COLUMNS
    # Exclude snapshot if it somehow remains as a column.
    dhw_columns = [
        col
        for col in dhw_desegregated_df.columns
        if col != "snapshot"
    ]
    # SUM ALL DHW LOAD COLUMNS ROW-BY-ROW
    total_dhw = dhw_desegregated_df[dhw_columns].sum(
        axis=1
    )
    # CREATE OUTPUT DATAFRAME
    DHW_load[country] = pd.DataFrame(
        {
            "DHW_load": total_dhw
        },
        index=dhw_desegregated_df.index
    )
    DHW_load[country].index.name = "snapshot"
    print(
        f"  ✅ {country}: "
        f"Summed {len(dhw_columns)} DHW columns."
    )
    print(
        f"  → {len(DHW_load[country])} snapshots."
    )
print(
    "\nDone! Created 'DHW_load' "
    "with one total DHW load time series per country."
)


--- Processing DHW Load: BE ---
  ✅ BE: Summed 10 DHW columns.
  → 8760 snapshots.

--- Processing DHW Load: FR ---
  ✅ FR: Summed 10 DHW columns.
  → 8760 snapshots.

--- Processing DHW Load: DE ---
  ✅ DE: Summed 10 DHW columns.
  → 8760 snapshots.

--- Processing DHW Load: NL ---
  ✅ NL: Summed 10 DHW columns.
  → 8760 snapshots.

--- Processing DHW Load: UK ---
  ✅ UK: Summed 20 DHW columns.
  → 8760 snapshots.

Done! Created 'DHW_load' with one total DHW load time series per country.


<div style= "background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099; " >
 <span style= "font-weight: bold; font-size: 16px; " >
Section J Overview: Target Year Adjustment & DHW Export
 </span >
 <div style= "border-top: 1px solid #000099; padding: 10px; " >
This section finalizes the DHW demand exports.<br> It adjusts the total DHW load time series to match the target year's temporal scope (handling leap years), renames the columns using the Sector identifiers from the VPP features CSV, and exports the finalized files to the dedicated VPP demand directory.
 </div >
 </div >
 <div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        J-1. Adjusting DHW_load to Target Year with Leap Year Handling
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process adjusts the <b>DHW_load</b> DataFrames to have the correct number of rows for the target year, handling <b>leap year</b> differences (February 29).<br>
        It ensures that all DHW load DataFrames have the exact number of timesteps expected for the target year.<br>
    <b>Adjustment Logic:</b>
            <div style="margin-left: 2em; font-size: 12px;">
        The script first determines if the target year is a leap year and calculates the expected number of rows (365 or 366 days × steps per day). For each zone, it:
                <div style="margin-left: 2em;">
        <b>If target year has more rows</b> (leap year) → Duplicate February 28 data to create a February 29 row.<br>
        <b>If target year has fewer rows</b> (non-leap year) → Remove February 29 data.
                </div>
        The process uses a <b>temporary leap year 2000 index</b> to safely handle February 29 during processing, then reindexes to the target year with a UTC timezone.<br>
        After adjustment, all DataFrames have the correct number of rows for the target year and are properly indexed with a UTC datetime index named "snapshot"
        </div>
    </div>
</div>

In [41]:
# TARGET YEAR SETTINGS
target_year = int(data_target_year)
is_leap_year = calendar.isleap(target_year)
# =============================================================================
steps_per_day = {
    "1h": 24,
    "30min": 48,
    "15min": 96
}
# =============================================================================
expected_rows = (
    366 if is_leap_year else 365
) * steps_per_day[data_target_time_step]
print(f"Target year   : {target_year}")
print(f"Leap year     : {is_leap_year}")
print(f"Expected rows : {expected_rows}")
# ADJUST ALL ZONES IN DHW_LOAD
for zone, df in DHW_load.items():
    df = df.copy()
    actual_rows = len(df)
    print(f"\nProcessing {zone}")
    print(f"Actual rows   : {actual_rows}")
    print(f"Expected rows : {expected_rows}")
    # CREATE TEMPORARY INDEX USING LEAP YEAR 2000
    # 2000 is a leap year, so Feb-29 can exist if needed.
    base_freq = {
        "1h": "h",
        "30min": "30min",
        "15min": "15min"
    }[data_target_time_step]
    temp_index = pd.date_range(
        start="2000-01-01 00:00:00",
        periods=actual_rows,
        freq=base_freq
    )
    df.index = temp_index
    # TARGET YEAR HAS MORE ROWS
    # ADD FEBRUARY 29
    if expected_rows > actual_rows:
        feb28_mask = (
            (temp_index.month == 2)
            & (temp_index.day == 28)
        )
        feb28 = df.loc[feb28_mask].copy()
        feb29_index = feb28.index + pd.Timedelta(days=1)
        feb29 = feb28.copy()
        feb29.index = feb29_index
        df = pd.concat(
            [df, feb29]
        )
        df = df.sort_index()
    # TARGET YEAR HAS FEWER ROWS
    # REMOVE FEBRUARY 29
    elif expected_rows < actual_rows:
        df = df[
            ~(
                (df.index.month == 2)
                & (df.index.day == 29)
            )
        ]
    # CREATE FINAL UTC INDEX
    final_index = pd.date_range(
        start=f"{target_year}-01-01 00:00:00",
        periods=len(df),
        freq=base_freq,
        tz="UTC"
    )
    df.index = final_index
    df.index.name = "snapshot"
    # SAVE BACK TO THE DICTIONARY
    DHW_load[zone] = df
    print(f"Final rows    : {len(df)}")
# FINAL MESSAGE
print(
    "\nDHW_load was adjusted "
    "to the target year."
)

Target year   : 2030
Leap year     : False
Expected rows : 8760

Processing BE
Actual rows   : 8760
Expected rows : 8760
Final rows    : 8760

Processing FR
Actual rows   : 8760
Expected rows : 8760
Final rows    : 8760

Processing DE
Actual rows   : 8760
Expected rows : 8760
Final rows    : 8760

Processing NL
Actual rows   : 8760
Expected rows : 8760
Final rows    : 8760

Processing UK
Actual rows   : 8760
Expected rows : 8760
Final rows    : 8760

DHW_load was adjusted to the target year.


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        J-2. Copying DHW Load Data to VPP Demand Directory
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process copies the <b>DHW_load</b> DataFrames to the <b>VPP demand directory</b>, renaming the columns using <b>Sector values</b> from the VPP features CSV.<br>
        The time step is automatically detected from the index, and the files are saved in the appropriate subfolder.<br>
    <b>Copy Logic:</b>
            <div style="margin-left: 2em; font-size: 12px;">
        For each zone, the script retrieves the DHW DataFrame and determines the time step from the index (1h, 30min, or 15min).<br>
        It then reads the VPP features CSV and extracts the <b>Sector</b> values.<br>
        The number of DHW columns is validated against the number of Sector values, and the DHW columns are <b>renamed</b> using the Sector values.<br>
        The renamed DataFrame is saved in the destination directory with the index included.<br>
        This ensures that DHW load data is correctly formatted for VPP demand profiles.
            </div>
    </div>
</div>

In [42]:
# COPY DHW LOAD DATA TO VPP DEMAND DIRECTORY
features_root = Path(vpp_features_data_folder_path)
demand_root = Path(vpp_demand_data_folder_path)
year_filename = f"{data_target_year}.csv"
for zone in zone_names:
    print(f"\nProcessing {zone}...")
    # GET DHW DATAFRAME
    dhw_df = DHW_load[zone].copy()
    print(f"  DHW columns: {len(dhw_df.columns)}")
    print(f"  DHW rows   : {len(dhw_df)}")
    # IDENTIFY TIME STEP FROM THE VERTICAL INDEX
    snapshot_index = pd.to_datetime(dhw_df.index)
    if len(snapshot_index) < 2:
        raise ValueError(
            f"{zone}: at least two timestamps are required "
            f"to identify the time step."
        )
    time_difference = (
        snapshot_index[1] - snapshot_index[0]
    )
    if time_difference == pd.Timedelta(hours=1):
        time_step = "1h"
    elif time_difference == pd.Timedelta(minutes=30):
        time_step = "30min"
    elif time_difference == pd.Timedelta(minutes=15):
        time_step = "15min"
    else:
        raise ValueError(
            f"{zone}: unsupported time step detected: "
            f"{time_difference}"
        )
    print(f"  Time step  : {time_step}")
    # READ FEATURES CSV
    features_file = (
        features_root
        / zone
        / year_filename
    )
    features_df = pd.read_csv(features_file)
    # EXTRACT SECTOR VALUES
    sector_values = (
        features_df["Sector"]
        .dropna()
        .astype(str)
        .tolist()
    )
    print(f"  Sector values: {len(sector_values)}")
    # CHECK NUMBER OF COLUMNS
    if len(dhw_df.columns) != len(sector_values):
        raise ValueError(
            f"{zone}: number of DHW columns does not match "
            f"number of Sector values.\n"
            f"  DHW columns   : {len(dhw_df.columns)}\n"
            f"  Sector values : {len(sector_values)}"
        )
    # RENAME DHW COLUMNS USING SECTOR VALUES
    dhw_df.columns = sector_values
    # RESTORE ORIGINAL INDEX AS THE FIRST CSV COLUMN
    dhw_df.index.name = "snapshot"
    # CREATE DESTINATION DIRECTORY
    destination_folder = (
        demand_root
        / zone
        / time_step
    )
    destination_folder.mkdir(
        parents=True,
        exist_ok=True
    )
    # DESTINATION FILE
    destination_file = (
        destination_folder
        / year_filename
    )
    # SAVE CSV
    dhw_df.to_csv(
        destination_file,
        index=True
    )
    print(f"  Saved to   : {destination_file}")


Processing BE...
  DHW columns: 1
  DHW rows   : 8760
  Time step  : 1h
  Sector values: 1
  Saved to   : /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario________VPP/VPP_data/VPP_demand/BE/1h/2030.csv

Processing FR...
  DHW columns: 1
  DHW rows   : 8760
  Time step  : 1h
  Sector values: 1
  Saved to   : /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario________VPP/VPP_data/VPP_demand/FR/1h/2030.csv

Processing DE...
  DHW columns: 1
  DHW rows   : 8760
  Time step  : 1h
  Sector values: 1
  Saved to   : /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario________VPP/VPP_data/VPP_demand/DE/1h/2030.csv

Processing NL...
  DHW columns: 1
  DHW rows   : 8760
  Time step  : 1h
  Sector values: 1
  Saved to   : /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario________VPP/VPP_data/VPP_demand/NL/1h/2030.csv

Processing UK...
  DHW columns: 1
  DHW rows   : 8760
  Time step  : 1h
  Sector values: 1
  Saved to   : /home/ray/Dispa-SET_Unleash/Datab